# RUL CPFL — baselines causais em Colab

Notebook **legível e reproduzível**, com núcleo compartilhado em `cer_rul.py` para comparar baselines tabulares de RUL no conjunto CPFL, com um único protocolo causal compartilhado por todos os rótulos, grades, candidatos e modelos.

- **Rótulos**: `dias-90`, `dias-ajustado`, `dias-FPT`, `porcentagem-data`, `porcentagem-registro`.
- **Equipamentos**: `A=[1,2]`, `B=[3]`, `C=[4,5]`, `D=[6,7]`.
- **Validação externa**: quatro folds leave-one-equipment-out.
- **Seleção**: somente pela MAE macro por componente no fold interno; o teste externo nunca escolhe configuração.
- **Modelos**: média constante de treino (referência sem peso), Ridge, SVR-RBF e ExtraTrees (128 árvores fixas e limitadas).
- **Política**: nenhum clamp de predição, nenhum MAPE, nenhum ranqueamento entre rótulos, nenhum vazamento temporal.

> Este notebook substitui o rascunho anterior. Ele foi escrito para ser lido e auditado: as decisões metodológicas aparecem em células de markdown e os artefatos de uma execução ficam em um único diretório de saída.

> **A execução científica no Colab ainda não foi verificada neste ambiente.** A verificação feita na criação cobriu apenas sintaxe, estrutura do notebook, esquema/consolidação dos CSVs e inspeção semântica com biblioteca padrão.


## Como usar

**Pré-requisito:** antes de executar qualquer célula, envie `cer_rul.py` pelo painel de arquivos do Colab para `/content`. Um runtime novo perde o arquivo, então ele deve ser reenviado a cada runtime; pode ser o mesmo `.py` do repositório.

1. No Colab, execute as células em ordem. A célula de configuração é a **única** célula de configuração do usuário.
2. O padrão é `run_training=False`: primeiro faça um *smoke test* que baixa/valida os dados, audita a limpeza, monta a EDA compartilhável, os endpoints e os splits, mostra a previsão de quantidade de ajustes e exporta um ZIP sem treinar nada.
3. Para treinar, altere somente `CONFIG["run_training"] = True` e **REEXECUTE TODAS AS CÉLULAS** (`Runtime > Restart and run all`). Como `EFFECTIVE_CONFIG` e `RUN_ID` dependem de `run_training`, reexecutar apenas a configuração e a última célula usa uma identidade obsoleta.
4. A saída padrão é persistente no Google Drive. O Drive é montado **antes** de qualquer `mkdir` ou escrita, inclusive quando a entrada é um ZIP/CSV enviado pelo usuário.
5. Se `CONFIG["input"]["mode"] = "auto"` e o diretório não existir, o notebook tenta os caminhos explícitos e, por último, o upload do Colab. A descoberta dos cinco CSVs é recursiva e precisa ser não ambígua.

## O que este protocolo faz — e o que ele não faz

- A limpeza remove duplicatas de `(Falha, Date)` apenas quando os oito sensores são idênticos. O alvo consolidado é a **média de todas as linhas originais**, inclusive valores repetidos; não há deduplicação prévia dos valores únicos de RUL.
- Os endpoints começam **depois de 30 registros únicos de aquecimento** de cada Falha, seguem com passo 3 e incluem sempre o último registro. Isso cobre o ciclo pós-aquecimento, não todas as linhas originais.
- Para cada endpoint `t`, as features usam somente registros da mesma Falha com data `<= t`. Não há `ffill`, interpolação, preenchimento com valor futuro ou mistura entre séries. A matriz tabular usa somente a janela regular de 32 bins; os valores brutos dos bins não entram nela.
- A representação `current` contém o último valor observado por sensor dentro da janela mais contagem/fração de ausência. A representação `summary` adiciona média, desvio e inclinação por **dia real** calculados sobre os valores finais dos bins observados da mesma janela.
- A seleção interna é feita **separadamente por família** (média, Ridge, SVR e ExtraTrees), sempre pela MAE macro por componente. A média é uma família sempre refeita e testada, não apenas uma candidata interna. O fold de teste externo é tocado uma única vez por família, após o refit da configuração escolhida dentro daquela família em todo o treino externo. Nenhuma família é escolhida como “vencedora” pelo teste externo.
- As datas FPT herdadas da célula 19 do notebook de origem servem somente para relatório de fase e para os pesos opcionais de treino. Elas não foram verificadas fisicamente de forma independente e não entram como features.

## Limitações estruturais

- Existem apenas quatro equipamentos e um único equipamento de validação interna por fold. A estimativa de seleção é, portanto, de baixa resolução.
- Cada fold deixa um equipamento inteiro de fora; os resultados descrevem generalização para um equipamento não visto, não para o mesmo equipamento no futuro.
- O número de componentes por equipamento é desigual: `A`, `C` e `D` têm duas Falhas cada; `B` tem uma. As métricas por componente não pressupõem contagens iguais.
- As métricas “overall” de um fold são calculadas **dentro do equipamento de teste** e, portanto, agregam os componentes presentes naquele equipamento. A chamada macro entre folds é uma média simples das quatro métricas de fold, com peso igual por equipamento, e **não** recombina as amostras dos folds. Ela não é uma métrica pooled entre equipamentos.
- A decisão de agrupar por equipamento físico foi fornecida no protocolo aprovado; ela não é inferida nem validada pelos dados deste notebook.
- SQLite em unidade de Drive não é seguro para múltiplos escritores simultâneos. Execute um único runtime por vez sobre o mesmo diretório de estudos.

## Reprodutibilidade

- Sementes fixas por fold/família, iguais entre candidatos da mesma família.
- Hashes de arquivos, dados limpos, endpoints, splits e identidade de execução são exportados.
- Cada JSON de execução é escrito atomicamente, contém as predições e identifica explicitamente a família; na retomada, a integridade é validada e um JSON sem predições é rejeitado.
- Caches e arquivos de entrada não são incluídos no ZIP de resultados por padrão.
- A célula `common-eda` é compartilhável: quantis dos oito sensores limpos e intervalos temporais por Falha, sem ajuste de modelo.


In [ ]:
# =============================================================================
# CÉLULA ÚNICA DE CONFIGURAÇÃO
# Altere apenas o que estiver indicado. O restante do notebook deriva caminhos,
# identidades, candidatos e contagens desta estrutura.
# IMPORTANTE: depois de alterar qualquer valor, reexecute TODAS as células
# (Runtime > Restart and run all). EFFECTIVE_CONFIG/RUN_ID são calculados em
# células posteriores e não podem ser atualizados executando só a última célula.
# =============================================================================

CONFIG = {
    # O padrão faz um smoke test completo, sem nenhum ajuste de modelo.
    # Para treinar: mude para True e reexecute o notebook desde esta célula.
    "run_training": False,

    "seed_base": 20261015,

    "input": {
        # "auto": tenta input_dir, depois caminhos explícitos, depois upload.
        # "drive_dir": exige input_dir. "files": usa somente zip_paths/csv_paths.
        "mode": "auto",
        "drive_dir": "/content/drive/MyDrive/Grupo2_Prognóstico/Resultados_preprocessamento_CPFL",
        "zip_paths": [],
        "csv_paths": [],
        "allow_colab_upload": True,
    },

    "output": {
        # "drive" é o padrão persistente. "local" emite aviso explícito de
        # efemeridade e só é permitido fora de um runtime com Drive montado.
        "mode": "drive",
        "drive_subdir": "RUL_CPFL_baselines",
        "local_dir": "/tmp/opencode/rul_cpfl_baselines",
        "download_zip": True,
        "local_only_acknowledged": False,
        "install_dependencies": False,
    },

    # Presets pequenos, não um produto cartesiano. Todos os 5 rótulos e os
    # 4 folds externos usam esta lista. O baseline de média é sempre incluído.
    # A largura padrão é 12 h; há alternativas correntes e de pré-processamento.
    "quick": {
        "presets": {
            "mean": [
                {},
            ],
            "ridge": [
                {"alpha": 1.0,  "bin_hours": 12, "representation": "summary",
                 "preprocessing": "standard", "timing": False, "weights": "none"},
                {"alpha": 10.0, "bin_hours": 12, "representation": "current",
                 "preprocessing": "robust", "timing": False, "weights": "component"},
            ],
            "svr": [
                {"C": 10.0, "epsilon": 0.10, "gamma": "scale",
                 "bin_hours": 12, "representation": "summary",
                 "preprocessing": "standard", "timing": False, "weights": "none"},
                {"C": 10.0, "epsilon": 0.05, "gamma": "scale",
                 "bin_hours": 12, "representation": "current",
                 "preprocessing": "minmax", "timing": False,
                 "weights": "component_phase"},
            ],
            "extra_trees": [
                {"max_depth": 8, "min_samples_leaf": 2, "max_features": "sqrt",
                 "bin_hours": 12, "representation": "summary",
                 "preprocessing": "signed_log1p_standard", "timing": False,
                 "weights": "none"},
                {"max_depth": 10, "min_samples_leaf": 1, "max_features": 0.5,
                 "bin_hours": 12, "representation": "current",
                 "preprocessing": "robust", "timing": False, "weights": "component"},
            ],
        },
    },

    # Busca opcional. O orçamento é de 15 tentativas TOTAIS por estudo
    # (família × rótulo × fold externo), não 15 por eixo. Ao retomar, o estudo
    # existente é validado e apenas as tentativas que faltam são executadas.
    "search": {
        "enabled": False,
        "n_trials": 15,
        "sampler_seed": 20261015,
        "families": ["ridge", "svr", "extra_trees"],
        "bin_hours_levels": [6, 12, 24],
        "representation_levels": ["current", "summary"],
        "preprocessing_levels": ["none", "standard", "robust", "minmax",
                                 "signed_log1p_standard"],
        "timing_levels": [False, True],
        "weights_levels": ["none", "component", "component_phase"],
        "spaces": {
            "ridge": {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
            "svr": {
                "C": [1.0, 10.0, 100.0],
                "epsilon": [0.01, 0.05, 0.10, 0.20],
                "gamma": ["scale", 0.01, 0.03, 0.10],
            },
            "extra_trees": {
                "max_depth": [4, 6, 8, 10, 12],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", 0.5, 1.0],
            },
        },
    },

    # Ablação opcional de UM eixo. Todos os níveis pré-declarados são
    # treinados/avaliados em cada fold externo; não há escolha de vencedor
    # pelo teste externo nem seleção entre níveis.
    "ablation": {
        "enabled": False,
        "axis": "representation",   # representation | timing | weights
        "levels": ["current", "summary"],
        "base_family": "ridge",
        "base_params": {"alpha": 1.0},
        "base_bin_hours": 12,
        "base_representation": "summary",
        "base_preprocessing": "standard",
        "base_timing": False,
        "base_weights": "none",
    },
}


MODEL_FAMILIES = ("mean", "ridge", "svr", "extra_trees")

def _describe_run_switch():
    if CONFIG["run_training"]:
        print("run_training=True: haverá ajuste de modelos e escrita de predições.")
    else:
        print("run_training=False (smoke): nenhum modelo será ajustado.")
        print("Para habilitar: CONFIG['run_training'] = True e reexecute TODAS as células.")
        print("Reexecutar só a configuração + última célula deixa RUN_ID obsoleto.")


_describe_run_switch()


In [ ]:
# =============================================================================
# SEÇÃO 1 — AMBIENTE, DRIVE E DIRETÓRIOS
# Nenhum mkdir/escrita de saída ocorre antes da montagem do Drive.
# =============================================================================

import os
import sys
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

OUTPUT_MODE = CONFIG["output"]["mode"]
DRIVE_MOUNTED = False

if OUTPUT_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError(
            "output.mode='drive' exige Google Colab. Use mode='local' apenas "
            "para um smoke local e leia o aviso de efemeridade."
        )
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / CONFIG["output"]["drive_subdir"]
elif OUTPUT_MODE == "local":
    OUTPUT_ROOT = Path(CONFIG["output"]["local_dir"])
    if IN_COLAB:
        print("AVISO: saída local no Colab é efêmera e pode desaparecer no restart.")
    else:
        print("AVISO EXPLÍCITO: modo local/efêmero; nada será persistido no Drive.")
        if not CONFIG["output"]["local_only_acknowledged"]:
            print("Defina output.local_only_acknowledged=True se esta for a intenção.")
else:
    raise ValueError("output.mode deve ser 'drive' ou 'local'.")

# Primeiro efeito de escrita só depois da montagem (ou do aviso local explícito).
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_ROOT / "cache"
STUDIES_DIR = CACHE_DIR / "studies"
INPUT_CACHE_DIR = CACHE_DIR / "inputs"
FEATURE_CACHE_DIR = CACHE_DIR / "features"
RESULTS_DIR = OUTPUT_ROOT / "results"
EXPORTS_DIR = OUTPUT_ROOT / "exports"
for _directory in (CACHE_DIR, STUDIES_DIR, INPUT_CACHE_DIR,
                   FEATURE_CACHE_DIR, RESULTS_DIR, EXPORTS_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

if CONFIG["output"]["install_dependencies"]:
    if not IN_COLAB:
        raise RuntimeError("Instalação opcional é permitida somente no Colab.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "pandas", "numpy", "scikit-learn", "matplotlib", "optuna"],
        check=True,
    )

# Núcleo compartilhado. No Colab, envie cer_rul.py para /content antes de
# executar o notebook. Localmente, inicie o Jupyter na raiz do repositório
# que contém cer_rul.py. Não há upload automático nem download do módulo.
try:
    import cer_rul as _cer_rul
except ModuleNotFoundError as exc:
    if exc.name != "cer_rul":
        raise
    raise ModuleNotFoundError(
        "cer_rul.py não encontrado. No Colab, envie cer_rul.py para /content "
        "(painel de arquivos ou google.colab.files.upload()). Localmente, "
        "inicie o Jupyter na raiz do repositório que contém cer_rul.py."
    ) from exc

RUNTIME_STARTED_UTC = datetime.now(timezone.utc).isoformat()
print("Runtime:", "Colab" if IN_COLAB else "fora do Colab")
print("Saída:", OUTPUT_ROOT)
print("Início UTC:", RUNTIME_STARTED_UTC)


In [ ]:
# =============================================================================
# SEÇÃO 1 (continuação) — BIBLIOTECAS CIENTÍFICAS
# Nenhum dado é ajustado nesta célula.
# =============================================================================

import importlib.metadata as importlib_metadata
import json
import math
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


def _installed_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


RUNTIME_VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": _installed_version("scikit-learn"),
    "matplotlib": matplotlib.__version__,
    "optuna": _installed_version("optuna"),
}
print(RUNTIME_VERSIONS)


In [ ]:
# =============================================================================
# SEÇÃO 2 — ESQUEMA, PROTOCOLO E CONSTANTES COMPARTILHADAS
# Tudo nesta célula é comum a qualquer notebook standalone que reutilize o
# mesmo protocolo causal (daí os IDs "common-*" das células no .ipynb).
# =============================================================================

PROTOCOL = {
    "name": "RUL CPFL baselines causais",
    "revision": "1.0.0",
    "warmup_records": 30,
    "endpoint_stride": 3,
    "context_bins": 32,
    "default_bin_hours": 12,
    "bin_hours_candidates": (6, 12, 24),
    "equipment_groups": {"A": (1, 2), "B": (3,), "C": (4, 5), "D": (6, 7)},
    "selection_metric": "inner_MACRO_COMPONENT_MAE_ONLY",
    "phase_rule": "saudavel se Date < FPT; pos_FPT se Date >= FPT",
    "fpt_annotations": {
        1: "2024-07-25", 2: "2024-07-25", 3: "2024-03-15",
        4: "2023-12-10", 5: "2023-12-10", 6: "2024-07-22", 7: "2024-07-22",
    },
    "fpt_note": (
        "Datas herdadas da célula 19 de RUL_CMS_TCN.ipynb, usadas apenas em "
        "relatório de fase e pesos opcionais de treino. Não foram verificadas "
        "fisicamente de forma independente."
    ),
    "labels": {
        "dias-90": {
            "filename": "Features_gerador_score_RUL-dias-90.csv",
            "kind": "days", "divisor": 100.0, "metric_factor": 1.0,
            "unit": "dias", "max_allowed": 100.0,
        },
        "dias-ajustado": {
            "filename": "Features_gerador_score_RUL-dias-ajustado.csv",
            "kind": "days", "divisor": 100.0, "metric_factor": 1.0,
            "unit": "dias", "max_allowed": 100.0,
        },
        "dias-FPT": {
            "filename": "Features_gerador_score_RUL-dias-FPT.csv",
            "kind": "days", "divisor": 100.0, "metric_factor": 1.0,
            "unit": "dias", "max_allowed": 100.0,
        },
        "porcentagem-data": {
            "filename": "Features_gerador_score_RUL-porcentagem-data.csv",
            "kind": "fraction", "divisor": 1.0, "metric_factor": 100.0,
            "unit": "pontos percentuais (pp)", "max_allowed": 1.0,
        },
        "porcentagem-registro": {
            "filename": "Features_gerador_score_RUL-porcentagem-registro.csv",
            "kind": "fraction", "divisor": 1.0, "metric_factor": 100.0,
            "unit": "pontos percentuais (pp)", "max_allowed": 1.0,
        },
    },
}

LABEL_ORDER = list(PROTOCOL["labels"].keys())
EQUIPMENT_ORDER = list(PROTOCOL["equipment_groups"].keys())
ID_TO_EQUIPMENT = {
    int(equipment_id): equipment_name
    for equipment_name, equipment_ids in PROTOCOL["equipment_groups"].items()
    for equipment_id in equipment_ids
}
FAULT_IDS = tuple(sorted(ID_TO_EQUIPMENT))

SENSOR_COLUMNS = (
    "g_0_RMS",
    "g_1_Meanfreq",
    "g_0_Variance",
    "g_0_Peak_peak",
    "e_Gerador Dianteiro _BPFO_2X|6332-M",
    "e_Gerador Dianteiro _BPFI_2X_1SB|6332-M",
    "e_Gerador Dianteiro _BPFI_1X_1SB|6332-M",
    "e_Gerador Dianteiro _BPFI_2X|6332-M",
)
SENSOR_KEYS = tuple(f"s{index}" for index in range(len(SENSOR_COLUMNS)))
SENSOR_BY_KEY = dict(zip(SENSOR_KEYS, SENSOR_COLUMNS))
CSV_COLUMNS = ("Date",) + SENSOR_COLUMNS + ("Falha", "RUL")

EXPECTED_RAW_ROWS = 2710
EXPECTED_CLEAN_KEYS = 1910
EXPECTED_TARGET_CONFLICTS = {
    "dias-90": 0,
    "dias-ajustado": 0,
    "dias-FPT": 0,
    "porcentagem-data": 0,
    "porcentagem-registro": 581,
}

# Proveniência fixada na criação. O arquivo-fonte é somente leitura.
SOURCE_PROVENANCE = {
    "notebook": "RUL_CMS_TCN.ipynb",
    "sha256": "7a78c1b225cb8e2309585679e31927ae02e0287d3dfdc6d10f310cdc7409636d",
    "inherited_cell": 19,
    "replaced_draft": "RUL_CPFL_experimentos_colab.ipynb",
    "replaced_draft_sha256": "53e619b80c45815ef78f8d6997ea51d8bd21112cee15076643aa37c0c42b613f",
}

REPRESENTATIONS = ("current", "summary")
PREPROCESSING_OPTIONS = ("none", "standard", "robust", "minmax",
                         "signed_log1p_standard")
WEIGHT_SCHEMES = ("none", "component", "component_phase")
N_ESTIMATORS = 128
FEATURE_CODE_VERSION = "causal-window-tabular-v2"
FORBIDDEN_INFERENCE_TOKENS = ("Falha", "Date", "RUL", "phase", "FPT",
                              "failure", "falha", "equipment")

MODEL_BOUNDS = {
    "ridge": {"alpha": (1e-6, 1e6)},
    "svr": {"C": (1e-6, 1e6), "epsilon": (0.0, 1e3)},
    "extra_trees": {"max_depth": (2, 16), "min_samples_leaf": (1, 50)},
}
SVR_EPSILON_UNITS_NOTE = (
    "epsilon do SVR é expresso na unidade do ALVO NORMALIZADO: RUL/100 para "
    "rótulos de dias e RUL (fração 0..1) para rótulos percentuais. Não é "
    "epsilon em dias nem em pontos percentuais."
)

print("Protocolo:", PROTOCOL["name"], PROTOCOL["revision"])
print("Rótulos:", LABEL_ORDER)
print("Sensores:", len(SENSOR_COLUMNS), "| equipamentos:", EQUIPMENT_ORDER)


In [ ]:
# =============================================================================
# SEÇÃO 3 — UTILITÁRIOS DE HASH, ESCRITA ATÔMICA E DESCOBERTA DE ENTRADA
# =============================================================================

import hashlib
import shutil
import stat as stat_module
import tempfile
import zipfile
from datetime import datetime as _datetime
from pathlib import PurePosixPath


def _json_default(value):
    if isinstance(value, (pd.Timestamp, _datetime)):
        return pd.Timestamp(value).isoformat()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, frozenset)):
        return sorted(value, key=lambda item: str(item))
    if isinstance(value, tuple):
        return list(value)
    raise TypeError(f"Objeto não serializável: {type(value)!r}")


def canonical_json(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
        allow_nan=False, default=_json_default,
    )


def sha256_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def sha256_text(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_bytes(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def atomic_write_text(path, text):
    atomic_write_bytes(path, str(text).encode("utf-8"))


def atomic_write_json(path, payload):
    atomic_write_text(path, canonical_json(payload) + "\n")


def read_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def file_sha_map(paths):
    return {key: sha256_file(Path(path)) for key, path in sorted(paths.items())}


def _safe_zip_member(info):
    name = info.filename.replace("\\", "/")
    pure = PurePosixPath(name)
    if pure.is_absolute() or not pure.parts:
        return None
    if any(part in ("", ".", "..") for part in pure.parts):
        return None
    if ":" in pure.parts[0]:
        return None
    mode = (info.external_attr >> 16) & 0o170000
    if stat_module.S_ISLNK(mode):
        raise ValueError(f"ZIP contém link simbólico: {info.filename!r}")
    return pure


def safe_extract_csv_zip(zip_path, destination_root=INPUT_CACHE_DIR):
    """Valida e extrai somente CSVs de um ZIP, sem path traversal."""
    zip_path = Path(zip_path)
    if zip_path.suffix.lower() != ".zip" or not zip_path.is_file():
        raise ValueError(f"ZIP inválido: {zip_path}")
    digest = sha256_file(zip_path)
    destination_root = Path(destination_root)
    destination_root.mkdir(parents=True, exist_ok=True)
    destination = destination_root / f"zip-{digest[:16]}"
    marker = destination / ".complete"
    if marker.exists():
        return destination
    if destination.exists():
        raise RuntimeError(
            f"Extração incompleta existe em {destination}; remova-a explicitamente."
        )

    temporary = Path(tempfile.mkdtemp(prefix=f".zip-{digest[:8]}-",
                                      dir=str(destination_root)))
    extracted = 0
    total_uncompressed = 0
    try:
        with zipfile.ZipFile(zip_path, "r") as archive:
            infos = archive.infolist()
            if len(infos) > 2000:
                raise ValueError("ZIP com entradas demais.")
            for info in infos:
                total_uncompressed += int(info.file_size)
                if total_uncompressed > 2 * 1024 ** 3:
                    raise ValueError("ZIP excede o limite descompactado de 2 GiB.")
                if info.flag_bits & 0x1:
                    raise ValueError("ZIP criptografado não é aceito.")
                if info.file_size > 1024 ** 3:
                    raise ValueError(f"Entrada grande demais: {info.filename!r}")
                if (info.compress_size > 0 and info.file_size > 50 * 1024 ** 2
                        and info.file_size / max(info.compress_size, 1) > 200):
                    raise ValueError(f"Razão de compressão suspeita: {info.filename!r}")
                member = _safe_zip_member(info)
                if member is None:
                    raise ValueError(f"Caminho inseguro no ZIP: {info.filename!r}")
                if info.is_dir():
                    continue
                if member.suffix.lower() != ".csv":
                    continue
                target = (temporary / member).resolve()
                if temporary.resolve() not in target.parents:
                    raise ValueError(f"Destino fora da extração: {member}")
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info, "r") as source, open(target, "wb") as out:
                    shutil.copyfileobj(source, out, length=1024 * 1024)
                extracted += 1
        if extracted == 0:
            raise ValueError("ZIP não contém CSV.")
        atomic_write_text(temporary / ".complete",
                          canonical_json({"zip_sha256": digest,
                                          "csv_files": extracted}) + "\n")
        temporary.rename(destination)
        return destination
    except Exception:
        shutil.rmtree(temporary, ignore_errors=True)
        raise


def _upload_colab_files():
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError("Upload vazio ou cancelado.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target_dir = INPUT_CACHE_DIR / "uploads" / stamp
    paths = []
    for original_name, payload in uploaded.items():
        safe_name = Path(original_name).name
        if not safe_name or safe_name != original_name:
            raise ValueError(f"Nome de upload inseguro: {original_name!r}")
        if Path(safe_name).suffix.lower() not in (".zip", ".csv"):
            raise ValueError(f"Upload deve ser ZIP ou CSV: {safe_name!r}")
        target = target_dir / safe_name
        atomic_write_bytes(target, payload)
        paths.append(target)
    return paths


def discover_input_files():
    """Descobre exatamente um CSV por rótulo, recursivamente e sem ambiguidade."""
    input_cfg = CONFIG["input"]
    mode = input_cfg["mode"]
    roots = []
    explicit = {}
    origins = []

    if mode not in ("auto", "drive_dir", "files"):
        raise ValueError("input.mode deve ser auto, drive_dir ou files.")

    if mode in ("auto", "drive_dir"):
        drive_dir = Path(input_cfg["drive_dir"]) if input_cfg["drive_dir"] else None
        if drive_dir is not None and drive_dir.is_dir():
            roots.append(drive_dir)
            origins.append(f"diretorio:{drive_dir.name}")
        elif mode == "drive_dir":
            raise FileNotFoundError(f"input.drive_dir não existe: {drive_dir}")

    for zip_name in input_cfg["zip_paths"]:
        extracted = safe_extract_csv_zip(Path(zip_name))
        roots.append(extracted)
        origins.append(f"zip:{Path(zip_name).name}")

    for csv_name in input_cfg["csv_paths"]:
        path = Path(csv_name)
        if not path.is_file():
            raise FileNotFoundError(f"CSV explícito não existe: {path}")
        if path.suffix.lower() != ".csv":
            raise ValueError(f"input.csv_paths contém não-CSV: {path}")
        matches = [label for label, meta in PROTOCOL["labels"].items()
                   if path.name == meta["filename"]]
        if len(matches) != 1:
            raise ValueError(f"CSV explícito não corresponde a um rótulo: {path.name}")
        label = matches[0]
        if label in explicit and explicit[label].resolve() != path.resolve():
            raise ValueError(f"Dois CSVs explícitos para {label}.")
        explicit[label] = path
        origins.append(f"csv:{path.name}")

    found = {label: set() for label in LABEL_ORDER}
    for path in explicit.values():
        label = next(lbl for lbl, meta in PROTOCOL["labels"].items()
                     if path.name == meta["filename"])
        found[label].add(path.resolve())

    def scan_roots():
        for root in roots:
            for label, meta in PROTOCOL["labels"].items():
                for path in Path(root).rglob(meta["filename"]):
                    if path.is_file():
                        found[label].add(path.resolve())

    scan_roots()

    missing = [label for label in LABEL_ORDER if not found[label]]
    if missing and mode == "auto" and input_cfg["allow_colab_upload"] and IN_COLAB:
        print("Entradas ausentes:", ", ".join(missing))
        print("Solicitando upload de ZIP(s)/CSV(s) no Colab.")
        for uploaded_path in _upload_colab_files():
            if uploaded_path.suffix.lower() == ".zip":
                roots.append(safe_extract_csv_zip(uploaded_path))
                origins.append(f"upload_zip:{uploaded_path.name}")
            else:
                matches = [label for label, meta in PROTOCOL["labels"].items()
                           if uploaded_path.name == meta["filename"]]
                if len(matches) != 1:
                    raise ValueError(
                        f"CSV enviado não corresponde a um rótulo: {uploaded_path.name}"
                    )
                label = matches[0]
                if label in explicit and explicit[label].resolve() != uploaded_path.resolve():
                    raise ValueError(f"Dois CSVs para {label} após upload.")
                explicit[label] = uploaded_path
                found[label].add(uploaded_path.resolve())
                origins.append(f"upload_csv:{uploaded_path.name}")
        scan_roots()

    missing = [label for label in LABEL_ORDER if not found[label]]
    if missing:
        raise FileNotFoundError("CSVs ausentes: " + ", ".join(missing))

    resolved = {}
    for label in LABEL_ORDER:
        candidates = sorted(found[label], key=lambda item: str(item))
        if len(candidates) != 1:
            raise RuntimeError(
                f"Descoberta ambígua para {label}: "
                + " | ".join(str(item) for item in candidates)
            )
        resolved[label] = candidates[0]
    return resolved, origins


print("Utilitários de I/O prontos.")


In [ ]:
# =============================================================================
# SEÇÃO 4 — LEITURA, VALIDAÇÃO, CONSOLIDAÇÃO E AUDITORIA
# A limpeza só agrupa dentro da mesma Falha e exige sensores idênticos.
# =============================================================================


coerce_fault_ids = _cer_rul.coerce_fault_ids


def load_raw_label(label, path):
    """Wrapper: ordem (label, path) -> núcleo (path, label) e constantes do notebook."""
    return _cer_rul.load_raw_label(
        path,
        label,
        expected_raw_rows=EXPECTED_RAW_ROWS,
        csv_columns=CSV_COLUMNS,
        sensor_columns=SENSOR_COLUMNS,
        fault_ids=FAULT_IDS,
        max_allowed=PROTOCOL["labels"][label]["max_allowed"],
        check_positive_sensors=True,
    )


def consolidate_frame(frame, label, enforce_expectations=True):
    """Wrapper: constantes do notebook e semântica original de enforce/fallback."""
    label_meta = PROTOCOL["labels"].get(label, {
        "divisor": 1.0, "metric_factor": 1.0, "unit": "unidade",
        "kind": "synthetic", "max_allowed": np.inf,
    })
    return _cer_rul.consolidate_frame(
        frame,
        label,
        sensor_columns=SENSOR_COLUMNS,
        csv_columns=CSV_COLUMNS,
        fault_ids=FAULT_IDS,
        id_to_equipment=ID_TO_EQUIPMENT,
        equipment_order=EQUIPMENT_ORDER,
        label_meta=label_meta,
        expected_clean_keys=EXPECTED_CLEAN_KEYS if enforce_expectations else None,
        expected_target_conflicts=(
            EXPECTED_TARGET_CONFLICTS[label] if enforce_expectations else None
        ),
        enforce_expectations=enforce_expectations,
    )


def clean_frame_fingerprint(clean):
    rows = []
    for record in clean.to_dict(orient="records"):
        rows.append([
            int(record["Falha"]),
            pd.Timestamp(record["Date"]).isoformat(),
            *[float(record[column]).hex() for column in SENSOR_COLUMNS],
            float(record["RUL"]).hex(),
            int(record["raw_count"]),
        ])
    return sha256_json(rows)


def key_fingerprint(clean):
    return sha256_json([
        [int(record["Falha"]), pd.Timestamp(record["Date"]).isoformat()]
        for record in clean.to_dict(orient="records")
    ])


def sensor_fingerprint(clean):
    return sha256_json([
        [int(record["Falha"]), pd.Timestamp(record["Date"]).isoformat(),
         *[float(record[column]).hex() for column in SENSOR_COLUMNS]]
        for record in clean.to_dict(orient="records")
    ])


def assert_raw_alignment(frames):
    reference_label = LABEL_ORDER[0]
    reference = frames[reference_label]
    for label in LABEL_ORDER[1:]:
        frame = frames[label]
        if len(frame) != len(reference):
            raise ValueError(f"Alinhamento bruto: {label} tem linhas diferentes.")
        if not frame.loc[:, ["Falha", "Date"]].equals(reference.loc[:, ["Falha", "Date"]]):
            raise ValueError(f"Alinhamento bruto: chaves/ordem divergem em {label}.")
        if not np.array_equal(
            frame.loc[:, SENSOR_COLUMNS].to_numpy(float),
            reference.loc[:, SENSOR_COLUMNS].to_numpy(float),
        ):
            raise ValueError(f"Alinhamento bruto: sensores divergem em {label}.")
    return True


def assert_clean_alignment(cleaned):
    reference_label = LABEL_ORDER[0]
    reference = cleaned[reference_label]
    for label in LABEL_ORDER[1:]:
        frame = cleaned[label]
        if len(frame) != len(reference):
            raise ValueError(f"Alinhamento limpo: {label} tem chaves diferentes.")
        if not frame.loc[:, ["Falha", "Date"]].equals(reference.loc[:, ["Falha", "Date"]]):
            raise ValueError(f"Alinhamento limpo: chaves/ordem divergem em {label}.")
        if not np.array_equal(
            frame.loc[:, SENSOR_COLUMNS].to_numpy(float),
            reference.loc[:, SENSOR_COLUMNS].to_numpy(float),
        ):
            raise ValueError(f"Alinhamento limpo: sensores divergem em {label}.")
    return True


def load_and_consolidate_all():
    files, origins = discover_input_files()
    raw = {label: load_raw_label(label, files[label]) for label in LABEL_ORDER}
    assert_raw_alignment(raw)
    cleaned = {}
    audits = {}
    terminals = {}
    for label in LABEL_ORDER:
        clean, audit, terminal = consolidate_frame(raw[label], label,
                                                   enforce_expectations=True)
        cleaned[label] = clean
        audits[label] = audit
        terminals[label] = terminal
    assert_clean_alignment(cleaned)
    hashes = file_sha_map(files)
    for label in LABEL_ORDER:
        print(
            f"{label}: raw={audits[label]['raw_rows']} | "
            f"chaves={audits[label]['raw_unique_keys']} | "
            f"conflitos_alvo={audits[label]['target_conflict_keys']} | "
            f"terminais!=0={audits[label]['terminal_nonzero_count']}"
        )
    return files, origins, raw, cleaned, audits, terminals, hashes


print("Funções de dados prontas.")


In [ ]:
# =============================================================================
# SEÇÃO 4 (execução) — CARGA E AUDITORIA DOS CINCO RÓTULOS
# =============================================================================

INPUT_FILES, INPUT_ORIGINS, RAW, CLEAN, AUDITS, TERMINALS, RAW_FILE_HASHES = (
    load_and_consolidate_all()
)

CLEAN_HASHES = {label: clean_frame_fingerprint(CLEAN[label]) for label in LABEL_ORDER}
KEY_HASH = key_fingerprint(CLEAN[LABEL_ORDER[0]])
SENSOR_HASH = sensor_fingerprint(CLEAN[LABEL_ORDER[0]])

DATA_BASE_IDENTITY = {
    "source_notebook_sha256": SOURCE_PROVENANCE["sha256"],
    "protocol_revision": PROTOCOL["revision"],
    "raw_file_sha256": RAW_FILE_HASHES,
    "clean_frame_sha256": CLEAN_HASHES,
    "key_sha256": KEY_HASH,
    "sensor_matrix_sha256": SENSOR_HASH,
}
DATA_BASE_HASH = sha256_json(DATA_BASE_IDENTITY)

AUDIT_ROWS = []
for label in LABEL_ORDER:
    audit = AUDITS[label]
    AUDIT_ROWS.append({
        "label": label,
        "raw_rows": audit["raw_rows"],
        "raw_unique_keys": audit["raw_unique_keys"],
        "raw_duplicate_keys": audit["raw_duplicate_keys"],
        "raw_max_multiplicity": audit["raw_max_multiplicity"],
        "exact_duplicate_groups": audit["exact_duplicate_groups"],
        "exact_duplicate_surplus_rows": audit["exact_duplicate_surplus_rows"],
        "target_conflict_keys": audit["target_conflict_keys"],
        "feature_conflict_keys": audit["feature_conflict_keys"],
        "clean_keys": audit["clean_keys"],
        "terminal_nonzero_count": audit["terminal_nonzero_count"],
        "raw_RUL_min": audit["raw_target_stats"]["min"],
        "raw_RUL_max": audit["raw_target_stats"]["max"],
        "clean_RUL_min": audit["clean_target_stats"]["min"],
        "clean_RUL_max": audit["clean_target_stats"]["max"],
    })
AUDIT_TABLE = pd.DataFrame(AUDIT_ROWS)
TERMINAL_TABLE = pd.concat(
    [TERMINALS[label].assign(label=label) for label in LABEL_ORDER],
    ignore_index=True,
)
TERMINAL_TABLE = TERMINAL_TABLE.loc[:, ["label", "Falha", "Date", "RUL"]]

print("Arquivos de entrada:")
for label in LABEL_ORDER:
    print(f"  {label}: {INPUT_FILES[label].name} | sha256={RAW_FILE_HASHES[label][:12]}…")
print("Origens:", ", ".join(INPUT_ORIGINS))
print("\nAuditoria bruta/limpa:")
display(AUDIT_TABLE)
print("\nAlvos terminais (nenhum zero foi imposto manualmente):")
display(TERMINAL_TABLE)
print("Chaves limpas totais por rótulo:", {label: len(CLEAN[label]) for label in LABEL_ORDER})
print("Hash base dos dados:", DATA_BASE_HASH)


In [ ]:
# =============================================================================
# CÉLULA EDA COMPARTILHÁVEL — SENSORES LIMPOS E CADÊNCIA POR FALHA
# Leve: não ajusta modelos e não depende de nenhum resultado de treino.
# =============================================================================


def build_sensor_quantiles(clean_reference):
    rows = []
    for column in SENSOR_COLUMNS:
        values = clean_reference[column].astype(float)
        rows.append({
            "sensor": column,
            "n": int(values.size),
            "min": float(values.min()),
            "q01": float(values.quantile(0.01)),
            "q05": float(values.quantile(0.05)),
            "q25": float(values.quantile(0.25)),
            "median": float(values.quantile(0.50)),
            "q75": float(values.quantile(0.75)),
            "q95": float(values.quantile(0.95)),
            "q99": float(values.quantile(0.99)),
            "max": float(values.max()),
            "mean": float(values.mean()),
            "std": float(values.std(ddof=0)),
        })
    return pd.DataFrame(rows)


def build_time_gap_table(clean_reference):
    rows = []
    for fault_id, group in clean_reference.groupby("Falha", sort=True):
        group = group.sort_values("Date", kind="mergesort")
        dates = group["Date"]
        gaps_h = dates.diff().dt.total_seconds().dropna() / 3600.0
        rows.append({
            "Falha": int(fault_id),
            "equipment": ID_TO_EQUIPMENT[int(fault_id)],
            "n_records": int(len(group)),
            "first_date": dates.iloc[0].isoformat(),
            "last_date": dates.iloc[-1].isoformat(),
            "duration_days": float(
                (dates.iloc[-1] - dates.iloc[0]).total_seconds() / 86400.0
            ),
            "median_gap_h": float(gaps_h.median()) if len(gaps_h) else None,
            "mean_gap_h": float(gaps_h.mean()) if len(gaps_h) else None,
            "min_gap_h": float(gaps_h.min()) if len(gaps_h) else None,
            "max_gap_h": float(gaps_h.max()) if len(gaps_h) else None,
            "q95_gap_h": float(gaps_h.quantile(0.95)) if len(gaps_h) else None,
        })
    return pd.DataFrame(rows)


# Os oito sensores e as chaves temporais são idênticos entre rótulos; a
# referência evita repetir exatamente a mesma EDA cinco vezes.
SENSOR_EDA = build_sensor_quantiles(CLEAN[LABEL_ORDER[0]])
TIME_GAP_EDA = build_time_gap_table(CLEAN[LABEL_ORDER[0]])

print("CÉLULA EDA COMPARTILHÁVEL — quantis dos oito sensores limpos:")
display(SENSOR_EDA)
print("Intervalos temporais por Falha (horas; datas reais):")
display(TIME_GAP_EDA)


In [ ]:
# =============================================================================
# SEÇÃO 5 — SPLITS EXPLÍCITOS: 4 FOLDS LEAVE-ONE-EQUIPMENT-OUT
# =============================================================================

def make_splits():
    return _cer_rul.make_splits(
        equipment_order=EQUIPMENT_ORDER,
        equipment_groups=PROTOCOL["equipment_groups"],
    )


SPLITS = make_splits()
SPLIT_HASH = sha256_json(SPLITS.to_dict(orient="records"))

# Invariantes do protocolo.
assert len(SPLITS) == 4
assert set(SPLITS["test_equipment"]) == set(EQUIPMENT_ORDER)
assert len(set(SPLITS["val_equipment"])) >= 1
for row in SPLITS.to_dict(orient="records"):
    test = set(row["test_ids"])
    val = set(row["val_ids"])
    inner = set(row["inner_train_ids"])
    outer = set(row["outer_train_ids"])
    assert test.isdisjoint(val | inner)
    assert val.isdisjoint(inner)
    assert val | inner == outer
    assert outer.isdisjoint(test)
    assert test | outer == set(FAULT_IDS)
    assert len(test) > 0 and len(val) > 0 and len(inner) > 0

print("Splits (um equipamento de teste e um único equipamento de validação interna):")
display(SPLITS.loc[:, ["fold_id", "test_ids", "val_ids", "inner_train_ids",
                       "outer_train_ids"]])
print("Hash dos splits:", SPLIT_HASH)


## Endpoints: ciclo pós-aquecimento, não o dataset inteiro

A lista de endpoints é construída **uma vez** a partir das chaves limpas e é reutilizada por todos os rótulos, larguras de grade, candidatos, modelos e níveis de ablação.

- Ordenação cronológica dentro de cada `Falha` (nunca entre Falhas).
- Aquecimento: os primeiros **30 registros únicos** não geram endpoint.
- Depois do aquecimento: índices `30, 33, 36, ...`, com passo 3.
- O **último registro** de cada Falha é sempre incluído, mesmo que o passo não caia nele.
- O hash da lista e um CSV com `endpoint_id`, Falha, equipamento, data, índice e contagem de contexto são exportados.

Isso significa que os resultados descrevem o ciclo **posterior ao aquecimento**. Nenhuma métrica deve ser apresentada como cobertura de todas as linhas originais.


In [ ]:
# =============================================================================
# SEÇÃO 6 — LISTA COMPARTILHADA DE ENDPOINTS
# =============================================================================

EXPECTED_ENDPOINT_COUNT = 574


def phase_for_date(date_value, fault_id):
    return _cer_rul.phase_for_date(
        date_value, fault_id,
        fpt_annotations=PROTOCOL["fpt_annotations"],
    )


def build_endpoint_rows(clean_reference, warmup=None, stride=None, enforce=True):
    return _cer_rul.build_endpoint_rows(
        clean_reference,
        warmup=(PROTOCOL["warmup_records"] if warmup is None else int(warmup)),
        stride=(PROTOCOL["endpoint_stride"] if stride is None else int(stride)),
        id_to_equipment=ID_TO_EQUIPMENT,
        fpt_annotations=PROTOCOL["fpt_annotations"],
        expected_endpoint_count=EXPECTED_ENDPOINT_COUNT,
        enforce=enforce,
    )


ENDPOINTS = build_endpoint_rows(CLEAN[LABEL_ORDER[0]])
ENDPOINT_HASH = sha256_json([
    [row.endpoint_id, int(row.Falha), pd.Timestamp(row.Date).isoformat(),
     int(row.record_index)]
    for row in ENDPOINTS.itertuples(index=False)
])
ENDPOINT_COUNTS = {
    str(int(fid)): int(count)
    for fid, count in ENDPOINTS.groupby("Falha").size().items()
}

TARGETS = {}
for label in LABEL_ORDER:
    target = ENDPOINTS.merge(
        CLEAN[label].loc[:, ["Falha", "Date", "RUL"]],
        on=["Falha", "Date"], how="left", sort=False, validate="one_to_one",
    )
    if target["RUL"].isna().any():
        raise ValueError(f"{label}: alvo ausente em algum endpoint.")
    if target["endpoint_id"].tolist() != ENDPOINTS["endpoint_id"].tolist():
        raise ValueError(f"{label}: ordem dos endpoints mudou no merge.")
    meta = PROTOCOL["labels"][label]
    target["label"] = label
    target["y_scaled"] = target["RUL"].astype(float) / float(meta["divisor"])
    target["y_metric"] = target["RUL"].astype(float) * float(meta["metric_factor"])
    target["y_unit"] = meta["unit"]
    TARGETS[label] = target

assert ENDPOINTS["Falha"].map(ID_TO_EQUIPMENT).equals(ENDPOINTS["equipment"])
assert all(len(TARGETS[label]) == len(ENDPOINTS) for label in LABEL_ORDER)
assert all(TARGETS[label]["endpoint_id"].tolist() == ENDPOINTS["endpoint_id"].tolist()
           for label in LABEL_ORDER)

print("Endpoints totais:", len(ENDPOINTS), "| por Falha:", ENDPOINT_COUNTS)
print("Aquecimento:", PROTOCOL["warmup_records"], "registros; passo:",
      PROTOCOL["endpoint_stride"], "; final sempre incluído.")
print("Hash dos endpoints:", ENDPOINT_HASH)
display(ENDPOINTS.groupby(["Falha", "equipment"], sort=True)
        .agg(endpoints=("endpoint_id", "size"),
             primeiro=("Date", "min"), ultimo=("Date", "max"))
        .reset_index())


## Features tabulares: somente a janela de 32 bins

A matriz tabular é deliberadamente compacta e **não recebe os 32 valores brutos por bin**:

- `current`: valor observado mais recente de cada sensor **dentro da janela de 32 bins**, contagem de registros por sensor, fração de bins vazios, número de registros na janela, bins observados e bins vazios.
- `summary`: `current` + média, desvio e inclinação por **dia real** calculados sobre os valores finais dos bins observados, usando seus tempos reais.
- `timing` opcional: duração fixa da janela (`32 × largura`), duração observada entre o primeiro e o último bin ocupado, gap do registro imediatamente anterior a `t` (que pode estar fora da grade) e mediana dos intervalos entre registros dentro da grade.

Toda estatística de valor usa somente os bins da janela `(t − 32·w, t]`. Registros antigos fora da grade não alteram `current` nem `summary`, embora possam aparecer apenas no gap anterior de timing. O contexto bruto dos bins não é adicionado à matriz; se um modelo Timestral precisar dele, deverá ser cacheado separadamente em uma tarefa própria.


In [ ]:
# =============================================================================
# SEÇÃO 7 — FEATURES CAUSAIS TABULARES
# A matriz tabular usa SOMENTE a janela regular de 32 bins ancorada à direita em t.
# Valores brutos por bin NÃO entram na matriz. O contexto bruto para um eventual
# modelo Timestral futuro deve ser cacheado separadamente, em outra tarefa.
# =============================================================================


def feature_columns(width_hours, representation, timing):
    return _cer_rul.feature_columns(
        width_hours, representation, timing,
        sensor_keys=SENSOR_KEYS,
        representations=REPRESENTATIONS,
        bin_hours_candidates=PROTOCOL["bin_hours_candidates"],
        forbidden_tokens=FORBIDDEN_INFERENCE_TOKENS,
    )


_daily_stats = _cer_rul.daily_stats


def _bin_stats(context_dates, context_values, endpoint_date, width_hours):
    return _cer_rul.bin_stats(
        context_dates, context_values, endpoint_date, width_hours,
        context_bins=PROTOCOL["context_bins"],
    )


def build_feature_row(endpoint_row, clean_group, width_hours):
    return _cer_rul.build_feature_row(
        endpoint_row, clean_group, width_hours,
        sensor_columns=SENSOR_COLUMNS,
        sensor_keys=SENSOR_KEYS,
        context_bins=PROTOCOL["context_bins"],
    )


def build_feature_frame(clean_reference, endpoints, width_hours):
    return _cer_rul.build_feature_frame(
        clean_reference, endpoints, width_hours,
        sensor_columns=SENSOR_COLUMNS,
        sensor_keys=SENSOR_KEYS,
        context_bins=PROTOCOL["context_bins"],
        bin_hours_candidates=PROTOCOL["bin_hours_candidates"],
        representations=REPRESENTATIONS,
        forbidden_tokens=FORBIDDEN_INFERENCE_TOKENS,
    )


def feature_cache_key(width_hours):
    return sha256_json({
        "feature_code_version": FEATURE_CODE_VERSION,
        "protocol_revision": PROTOCOL["revision"],
        "data_base_hash": DATA_BASE_HASH,
        "endpoint_hash": ENDPOINT_HASH,
        "width_hours": int(width_hours),
        "n_bins": PROTOCOL["context_bins"],
        "sensor_keys": list(SENSOR_KEYS),
    })


FEATURE_MATRICES = {}


def get_feature_matrix(width_hours):
    width_hours = int(width_hours)
    if width_hours in FEATURE_MATRICES:
        return FEATURE_MATRICES[width_hours]
    key = feature_cache_key(width_hours)
    path = FEATURE_CACHE_DIR / f"{key}.npz"
    expected_columns = feature_columns(width_hours, "summary", True)
    if path.exists():
        with np.load(path, allow_pickle=False) as cached:
            stored_key = str(cached["cache_key"].item())
            stored_columns = json.loads(str(cached["columns"].item()))
            stored_ids = [str(value) for value in cached["endpoint_ids"].tolist()]
            if stored_key != key:
                raise RuntimeError("Cache de features com identidade incompatível.")
            if stored_columns != expected_columns:
                raise RuntimeError("Cache de features com colunas incompatíveis.")
            if stored_ids != ENDPOINTS["endpoint_id"].tolist():
                raise RuntimeError("Cache de features com endpoints incompatíveis.")
            matrix = pd.DataFrame(
                np.asarray(cached["X"], dtype=float),
                index=pd.Index(stored_ids, name="endpoint_id"),
                columns=stored_columns,
            )
        print(f"Features {width_hours} h: cache válido ({matrix.shape}).")
    else:
        matrix = build_feature_frame(CLEAN[LABEL_ORDER[0]], ENDPOINTS, width_hours)
        temporary = path.with_name(f".{path.stem}.tmp.npz")
        np.savez_compressed(
            temporary,
            X=matrix.to_numpy(float),
            columns=np.array(json.dumps(matrix.columns.tolist())),
            endpoint_ids=np.array(matrix.index.tolist()),
            cache_key=np.array(key),
        )
        os.replace(temporary, path)
        print(f"Features {width_hours} h: construídas ({matrix.shape}) e cacheadas.")
    if matrix.isna().all(axis=0).any():
        print(f"  colunas totalmente vazias em {width_hours} h:",
              int(matrix.isna().all(axis=0).sum()))
    FEATURE_MATRICES[width_hours] = matrix
    return matrix


print("Funções de features prontas. Dimensão summary/current por largura:")
for _width in PROTOCOL["bin_hours_candidates"]:
    print(f"  {_width} h: summary={len(feature_columns(_width, 'summary', True))} | "
          f"current={len(feature_columns(_width, 'current', True))}")


## Modelos e pré-processamento

- **Referência**: média constante do alvo de treino, **sem pesos** e tratada como família independente; é sempre refeita e testada, não apenas uma candidata interna.
- **Ridge**: regressão linear regularizada.
- **SVR-RBF**: `epsilon` está na unidade do alvo normalizado (`RUL/100` para dias; fração 0..1 para percentuais). Isso é explicitado para evitar a leitura errada de “epsilon em dias”.
- **ExtraTreesRegressor**: exatamente `128` árvores, profundidade e folha limitadas; não há Random Forest, TCN, ARIMA nem tuning fora do espaço declarado.
- Cada família tem sua própria configuração interna, refit externo e predições. Não há seleção de uma família vencedora pelo teste externo.
- **Pré-processamento**: `none`, `standard`, `robust`, `minmax` e `signed_log1p_standard`.
  - `signed_log1p_standard` aplica `sign(x) * log1p(abs(x))` e depois StandardScaler. **Não** é `log` simples, pois inclinações podem ser negativas.
- O imputador e o scaler são aprendidos **somente** no treino da etapa:
  - treino interno (2 equipamentos) para selecionar;
  - treino externo (3 equipamentos) para o refit final.
- Toda coluna base recebe um indicador fixo de ausência (`na__<coluna>`). Colunas totalmente vazias no treino recebem uma constante aprendida (`0.0`), e não uma estatística impossível.
- A forma da matriz não depende do fold: os indicadores são definidos pelo esquema, não pelo conteúdo de um fold específico.
- Pesos opcionais (`component`, `component_phase`) são calculados apenas no treino corrente; validação, teste e a referência de média permanecem sem peso.


In [ ]:
# =============================================================================
# SEÇÃO 8 — IMPUTAÇÃO/ESCALA APRENDIDAS, CANDIDATOS E MODELOS
# =============================================================================

STRUCTURAL_KEYS = {"bin_hours", "representation", "preprocessing", "timing", "weights"}


def signed_log1p(values):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.log1p(np.abs(values))


class TabularPreprocessor:
    """Imputação + indicadores fixos + escala, ajustados somente no treino."""

    def __init__(self, method):
        if method not in PREPROCESSING_OPTIONS:
            raise ValueError(f"Pré-processamento inválido: {method}")
        self.method = method

    def fit(self, frame):
        if not isinstance(frame, pd.DataFrame) or frame.shape[1] == 0:
            raise ValueError("Pré-processador exige DataFrame com colunas.")
        if frame.columns.duplicated().any():
            raise ValueError("Colunas duplicadas no pré-processador.")
        self.columns_ = list(frame.columns)
        self.indicator_columns_ = [f"na__{column}" for column in self.columns_]
        self.fill_values_ = {}
        self.constant_fallback_columns_ = []
        for column in self.columns_:
            values = pd.to_numeric(frame[column], errors="coerce").to_numpy(float)
            finite = values[np.isfinite(values)]
            if finite.size:
                fill = float(np.median(finite))
            else:
                fill = 0.0
                self.constant_fallback_columns_.append(column)
            self.fill_values_[column] = fill
        base = self._base_array(frame)
        base = self._transform_values(base)
        if self.method in ("standard", "robust", "minmax", "signed_log1p_standard"):
            if self.method in ("standard", "signed_log1p_standard"):
                self.scaler_ = StandardScaler()
            elif self.method == "robust":
                self.scaler_ = RobustScaler()
            else:
                self.scaler_ = MinMaxScaler()
            self.scaler_.fit(base)
        else:
            self.scaler_ = None
        self.n_features_in_ = len(self.columns_)
        self.n_features_out_ = len(self.columns_) + len(self.indicator_columns_)
        return self

    def _base_array(self, frame):
        if list(frame.columns) != self.columns_:
            raise ValueError("Colunas diferentes do ajuste do pré-processador.")
        columns = []
        for column in self.columns_:
            values = pd.to_numeric(frame[column], errors="coerce").to_numpy(float)
            if column in self.constant_fallback_columns_:
                values = np.full(values.shape, self.fill_values_[column], dtype=float)
            else:
                values = np.where(np.isfinite(values), values,
                                  self.fill_values_[column])
            columns.append(values)
        return np.column_stack(columns).astype(float)

    def _transform_values(self, base):
        if self.method == "signed_log1p_standard":
            return signed_log1p(base)
        return base

    def transform(self, frame):
        base = self._base_array(frame)
        base = self._transform_values(base)
        if self.scaler_ is not None:
            base = self.scaler_.transform(base)
        missing = np.column_stack([
            (~np.isfinite(pd.to_numeric(frame[column], errors="coerce")
                          .to_numpy(float))).astype(float)
            for column in self.columns_
        ])
        result = np.column_stack([base, missing])
        if not np.isfinite(result).all():
            raise ValueError("Pré-processador produziu valores não finitos.")
        return result

    def fit_transform(self, frame):
        return self.fit(frame).transform(frame)

    def get_feature_names_out(self):
        return np.asarray(self.columns_ + self.indicator_columns_, dtype=object)


def _validate_param_range(name, value, lower, upper):
    if not isinstance(value, (int, float, np.integer, np.floating)):
        raise ValueError(f"{name} deve ser numérico.")
    value = float(value)
    if not np.isfinite(value) or not (lower <= value <= upper):
        raise ValueError(f"{name}={value} fora de [{lower}, {upper}].")
    return value


def validate_model_params(family, params):
    params = dict(params)
    if family == "mean":
        if params:
            raise ValueError("A referência de média não aceita parâmetros de modelo.")
        return {}
    if family == "ridge":
        if set(params) != {"alpha"}:
            raise ValueError("Ridge exige exatamente alpha.")
        _validate_param_range("alpha", params["alpha"], *MODEL_BOUNDS["ridge"]["alpha"])
        return {"alpha": float(params["alpha"])}
    if family == "svr":
        if set(params) != {"C", "epsilon", "gamma"}:
            raise ValueError("SVR exige C, epsilon e gamma.")
        _validate_param_range("C", params["C"], *MODEL_BOUNDS["svr"]["C"])
        _validate_param_range("epsilon", params["epsilon"],
                              *MODEL_BOUNDS["svr"]["epsilon"])
        gamma = params["gamma"]
        if gamma != "scale":
            _validate_param_range("gamma", gamma, 1e-9, 1e6)
            gamma = float(gamma)
        return {"C": float(params["C"]), "epsilon": float(params["epsilon"]),
                "gamma": gamma}
    if family == "extra_trees":
        if set(params) != {"max_depth", "min_samples_leaf", "max_features"}:
            raise ValueError("ExtraTrees exige max_depth, min_samples_leaf e max_features.")
        depth = int(params["max_depth"])
        leaf = int(params["min_samples_leaf"])
        if depth != params["max_depth"] or leaf != params["min_samples_leaf"]:
            raise ValueError("max_depth e min_samples_leaf devem ser inteiros.")
        _validate_param_range("max_depth", depth, *MODEL_BOUNDS["extra_trees"]["max_depth"])
        _validate_param_range("min_samples_leaf", leaf,
                              *MODEL_BOUNDS["extra_trees"]["min_samples_leaf"])
        max_features = params["max_features"]
        if max_features not in ("sqrt", 0.5, 1.0):
            raise ValueError("max_features deve ser 'sqrt', 0.5 ou 1.0.")
        return {"max_depth": depth, "min_samples_leaf": leaf,
                "max_features": max_features}
    raise ValueError(f"Família desconhecida: {family}")


def seed_for(fold_id, family):
    if family == "mean":
        return 0
    return int(sha256_json({
        "seed_base": int(CONFIG["seed_base"]),
        "fold": str(fold_id),
        "family": str(family),
    })[:8], 16) % (2 ** 31 - 1)


def candidate_id(candidate):
    payload = {key: candidate[key] for key in
               ("family", "params", "bin_hours", "representation",
                "preprocessing", "timing", "weights")}
    payload["origin"] = candidate.get("origin", "quick")
    return sha256_json(payload)[:12]


def candidate_from_preset(family, preset, fold_id, preset_index):
    preset = dict(preset)
    if family == "mean":
        if preset:
            raise ValueError("Preset da média deve ser vazio.")
        params = {}
        structure = {"bin_hours": int(PROTOCOL["default_bin_hours"]),
                     "representation": "current", "preprocessing": "none",
                     "timing": False, "weights": "none"}
    else:
        if "bin_hours" not in preset:
            raise ValueError("Preset sem bin_hours.")
        structure = {
            "bin_hours": int(preset["bin_hours"]),
            "representation": preset["representation"],
            "preprocessing": preset["preprocessing"],
            "timing": bool(preset["timing"]),
            "weights": preset["weights"],
        }
        params = {key: value for key, value in preset.items()
                  if key not in STRUCTURAL_KEYS}
    params = validate_model_params(family, params)
    if structure["bin_hours"] not in PROTOCOL["bin_hours_candidates"]:
        raise ValueError("Largura de grade não aprovada.")
    if structure["representation"] not in REPRESENTATIONS:
        raise ValueError("Representação não aprovada.")
    if structure["preprocessing"] not in PREPROCESSING_OPTIONS:
        raise ValueError("Pré-processamento não aprovado.")
    if structure["weights"] not in WEIGHT_SCHEMES:
        raise ValueError("Esquema de pesos não aprovado.")
    candidate = {
        "family": family,
        "params": params,
        **structure,
        "seed": seed_for(fold_id, family),
        "origin": f"quick:{family}:{preset_index}",
        "fold_id": fold_id,
    }
    candidate["candidate_id"] = candidate_id(candidate)
    return candidate


def quick_candidates(fold_id):
    candidates = []
    for family in MODEL_FAMILIES:
        presets = CONFIG["quick"]["presets"].get(family, [])
        for preset_index, preset in enumerate(presets):
            candidates.append(
                candidate_from_preset(family, preset, fold_id, preset_index)
            )
    if not any(candidate["family"] == "mean" for candidate in candidates):
        raise ValueError("O baseline de média é obrigatório.")
    return candidates


class FittedModel:
    def __init__(self, candidate, preprocessor=None, estimator=None,
                 constant=None, fit_rows=None):
        self.candidate = candidate
        self.preprocessor = preprocessor
        self.estimator = estimator
        self.constant = constant
        self.fit_rows = fit_rows

    def predict(self, frame):
        if self.constant is not None:
            return np.full(len(frame), float(self.constant), dtype=float)
        matrix = self.preprocessor.transform(frame)
        return np.asarray(self.estimator.predict(matrix), dtype=float)


def fit_and_predict(candidate, X_train, y_train, X_eval, train_ids, train_phases,
                    counter=None, purpose="selection"):
    if counter is not None:
        counter["model_fits"] = counter.get("model_fits", 0) + 1
        counter[f"{purpose}_fits"] = counter.get(f"{purpose}_fits", 0) + 1
        if candidate["family"] == "mean":
            counter["baseline_fits"] = counter.get("baseline_fits", 0) + 1
            counter[f"{purpose}_baseline_fits"] = (
                counter.get(f"{purpose}_baseline_fits", 0) + 1
            )
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) == 0:
        raise ValueError("Treino vazio.")
    if not np.isfinite(y_train).all():
        raise ValueError("Alvo de treino não finito.")

    if candidate["family"] == "mean":
        constant = float(np.mean(y_train))  # sem pesos, por definição
        model = FittedModel(candidate, constant=constant, fit_rows=int(len(y_train)))
        return model.predict(X_eval), model, {"n_features_out": 0, "constant": constant}

    preprocessor = TabularPreprocessor(candidate["preprocessing"])
    matrix_train = preprocessor.fit_transform(X_train)
    matrix_eval = preprocessor.transform(X_eval)
    weights = compute_weights(train_ids, train_phases, candidate["weights"])
    family = candidate["family"]
    params = candidate["params"]
    if family == "ridge":
        estimator = Ridge(alpha=params["alpha"], fit_intercept=True)
    elif family == "svr":
        estimator = SVR(
            kernel="rbf", C=params["C"], epsilon=params["epsilon"],
            gamma=params["gamma"], cache_size=1024,
        )
    elif family == "extra_trees":
        estimator = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=int(candidate["seed"]),
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Família sem implementação: {family}")
    estimator.fit(matrix_train, y_train, sample_weight=weights)
    model = FittedModel(candidate, preprocessor=preprocessor, estimator=estimator,
                        constant=None, fit_rows=int(len(y_train)))
    prediction = model.predict(X_eval)
    metadata = {
        "n_features_in": int(len(preprocessor.columns_)),
        "n_features_out": int(preprocessor.n_features_out_),
        "n_missing_indicators": int(len(preprocessor.indicator_columns_)),
        "constant_fallback_columns": list(preprocessor.constant_fallback_columns_),
        "weight_min": float(np.min(weights)),
        "weight_max": float(np.max(weights)),
        "weight_mean": float(np.mean(weights)),
        "seed": int(candidate["seed"]),
    }
    return prediction, model, metadata


print("Modelos disponíveis:", MODEL_FAMILIES)
print("SVR:", SVR_EPSILON_UNITS_NOTE)


In [ ]:
# =============================================================================
# SEÇÃO 9 — PESOS, MÉTRICAS E AGREGAÇÃO MACRO ENTRE FOLDS
# =============================================================================


def compute_weights(ids, phases, scheme):
    return _cer_rul.compute_weights(ids, phases, scheme,
                                    weight_schemes=WEIGHT_SCHEMES)


regression_metrics = _cer_rul.regression_metrics
metric_breakdown = _cer_rul.metric_breakdown


def selection_objective(y_true_metric, y_pred_metric, ids):
    y_true_metric = np.asarray(y_true_metric, dtype=float)
    y_pred_metric = np.asarray(y_pred_metric, dtype=float)
    ids = np.asarray(ids, dtype=int)
    component_mae = {}
    for fault_id in sorted(set(int(value) for value in ids)):
        mask = ids == fault_id
        component_mae[str(fault_id)] = regression_metrics(
            y_true_metric[mask], y_pred_metric[mask])["mae"]
    score = float(np.mean(list(component_mae.values())))
    return score, component_mae


def scaled_prediction_to_units(prediction_scaled, label):
    return _cer_rul.scaled_prediction_to_units(
        prediction_scaled, label, label_meta=PROTOCOL["labels"])


def plot_metric_columns(label):
    """Colunas e unidade usadas nos gráficos: y_metric é dias ou pp."""
    if label not in PROTOCOL["labels"]:
        raise ValueError(f"Rótulo desconhecido: {label}")
    return "y_metric", "pred_metric", PROTOCOL["labels"][label]["unit"]


def predictions_frame(target_subset, prediction_scaled, label, fold, candidate):
    meta = PROTOCOL["labels"][label]
    if len(target_subset) != len(prediction_scaled):
        raise ValueError("Predição e alvo desalinhados.")
    native, metric = scaled_prediction_to_units(prediction_scaled, label)
    frame = pd.DataFrame({
        "label": label,
        "unit": meta["unit"],
        "fold_id": fold["fold_id"],
        "test_equipment": fold["test_equipment"],
        "endpoint_id": target_subset["endpoint_id"].tolist(),
        "Falha": target_subset["Falha"].astype(int).tolist(),
        "equipment": target_subset["equipment"].astype(str).tolist(),
        "Date": pd.to_datetime(target_subset["Date"]).tolist(),
        "phase": target_subset["phase"].astype(str).tolist(),
        "candidate_id": candidate["candidate_id"],
        "family": candidate["family"],
        "y_native": target_subset["RUL"].astype(float).to_numpy(),
        "pred_native": native,
        "y_metric": target_subset["y_metric"].astype(float).to_numpy(),
        "pred_metric": metric,
        "residual_metric": metric - target_subset["y_metric"].astype(float).to_numpy(),
        "split_role": "outer_test",
    })
    if not np.isfinite(frame["pred_metric"].to_numpy(float)).all():
        raise ValueError("Predição não finita; não há clamp.")
    return frame


def predictions_to_json(frame):
    records = []
    for row in frame.itertuples(index=False):
        records.append({
            "label": row.label,
            "unit": row.unit,
            "fold_id": row.fold_id,
            "test_equipment": row.test_equipment,
            "endpoint_id": row.endpoint_id,
            "Falha": int(row.Falha),
            "equipment": row.equipment,
            "Date": pd.Timestamp(row.Date).isoformat(),
            "phase": row.phase,
            "candidate_id": row.candidate_id,
            "family": row.family,
            "y_native": float(row.y_native),
            "pred_native": float(row.pred_native),
            "y_metric": float(row.y_metric),
            "pred_metric": float(row.pred_metric),
            "residual_metric": float(row.residual_metric),
            "split_role": row.split_role,
        })
    return records


def fold_metric_rows(prediction_frame):
    return _cer_rul.fold_metric_rows(
        prediction_frame, label_meta=PROTOCOL["labels"])


aggregate_metric_rows = _cer_rul.aggregate_metric_rows




print("Funções de métricas prontas; bias = predição - real; sem clamp e sem MAPE.")


In [ ]:
# =============================================================================
# SEÇÃO 10 — IDENTIDADES, RESUME ATÔMICO E INTEGRIDADE DE PREDIÇÕES
# =============================================================================

class RunIntegrityError(RuntimeError):
    pass


EFFECTIVE_CONFIG = {
    "run_training": bool(CONFIG["run_training"]),
    "seed_base": int(CONFIG["seed_base"]),
    "quick": CONFIG["quick"],
    "search": CONFIG["search"],
    "ablation": CONFIG["ablation"],
    "code_revision": CONFIG.get("code_revision", "rul-cpfl-baselines-v1"),
    "feature_code_version": FEATURE_CODE_VERSION,
    "n_estimators": N_ESTIMATORS,
    "model_bounds": MODEL_BOUNDS,
    "preprocessing_options": PREPROCESSING_OPTIONS,
    "weight_schemes": WEIGHT_SCHEMES,
    "model_families": MODEL_FAMILIES,
    "model_specific": CONFIG.get("model_specific", {}),
}
EFFECTIVE_CONFIG_HASH = sha256_json(EFFECTIVE_CONFIG)
PROTOCOL_HASH = sha256_json(PROTOCOL)
DATA_IDENTITY = {
    **DATA_BASE_IDENTITY,
    "endpoint_sha256": ENDPOINT_HASH,
    "split_sha256": SPLIT_HASH,
}
DATA_IDENTITY_HASH = sha256_json(DATA_IDENTITY)
RUN_ID = sha256_json({
    "effective_config": EFFECTIVE_CONFIG,
    "protocol": PROTOCOL,
    "data_identity": DATA_IDENTITY,
})[:16]
RUN_DIR = RESULTS_DIR / RUN_ID
RUNS_DIR = RUN_DIR / "runs"
ABLATION_DIR = RUN_DIR / "ablation"
PLOTS_DIR = RUN_DIR / "plots"

RUN_REQUIRED_KEYS = (
    "run_identity", "label", "family", "fold_id", "candidate_mode",
    "chosen_candidate", "metrics", "counts", "predictions",
)


def run_identity_for(label, fold, family, candidate_mode, expected_prediction_keys):
    return sha256_json({
        "run_id": RUN_ID,
        "label": label,
        "family": family,
        "fold_id": fold["fold_id"],
        "candidate_mode": candidate_mode,
        "test_equipment": fold["test_equipment"],
        "prediction_keys_sha256": sha256_json(sorted(expected_prediction_keys)),
    })


def run_file_path(label, fold_id, family):
    return RUNS_DIR / f"{label}__{fold_id}__{family}.json"


def seal_payload(payload):
    sealed = json.loads(canonical_json(payload))
    sealed["complete"] = True
    sealed.pop("integrity_sha256", None)
    sealed["integrity_sha256"] = sha256_json(sealed)
    return sealed


def save_complete_run(path, payload):
    sealed = seal_payload(payload)
    if not sealed.get("predictions"):
        raise RunIntegrityError("Recusando gravar execução sem predições.")
    atomic_write_json(path, sealed)
    return sealed


def load_complete_run(path, expected_identity, expected_prediction_keys):
    path = Path(path)
    if not path.exists():
        return None
    payload = read_json(path)
    for key in RUN_REQUIRED_KEYS:
        if key not in payload:
            raise RunIntegrityError(f"Execução incompleta, campo ausente: {key}")
    if payload.get("complete") is not True:
        raise RunIntegrityError("Execução sem marcador complete=True.")
    if "predictions" not in payload or not isinstance(payload["predictions"], list):
        raise RunIntegrityError("Execução completa sem lista de predições.")
    stored_integrity = payload.get("integrity_sha256")
    recomputed = dict(payload)
    recomputed.pop("integrity_sha256", None)
    if stored_integrity != sha256_json(recomputed):
        raise RunIntegrityError("Falha de integridade do JSON de execução.")
    if payload.get("run_identity") != expected_identity:
        raise RunIntegrityError(
            "Identidade incompatível para o mesmo caminho de execução."
        )
    prediction_ids = [record.get("endpoint_id") for record in payload["predictions"]]
    if any(identifier is None for identifier in prediction_ids):
        raise RunIntegrityError("Predição sem endpoint_id.")
    if len(prediction_ids) != len(set(prediction_ids)):
        raise RunIntegrityError("Predições duplicadas no JSON.")
    if set(prediction_ids) != set(expected_prediction_keys):
        raise RunIntegrityError(
            "Predições ausentes/extras em relação ao teste externo esperado."
        )
    return payload


print("RUN_ID:", RUN_ID)
print("Config efetiva:", EFFECTIVE_CONFIG_HASH[:16],
      "| dados:", DATA_IDENTITY_HASH[:16])


In [ ]:
# =============================================================================
# SEÇÃO 11 — BUSCA OPTUNA OPCIONAL (15 TENTATIVAS TOTAIS POR ESTUDO)
# Escopo de cada estudo: família × rótulo × fold externo.
# Os eixos de grade/pré-processamento/representação/pesos são escolhidos
# apenas no fold interno. Nenhum trial olha o teste externo.
# =============================================================================

# SQLite fica em cache/studies dentro do diretório único de saída.
# Execute um único runtime por vez: SQLite em Drive não suporta múltiplos
# escritores concorrentes com segurança.
SEARCH_AXES = ("bin_hours_levels", "representation_levels",
               "preprocessing_levels", "timing_levels", "weights_levels")


def search_signature(label, fold, family):
    search_cfg = CONFIG["search"]
    payload = {
        "run_id": RUN_ID,
        "label": label,
        "fold": fold,
        "family": family,
        "n_trials": int(search_cfg["n_trials"]),
        "families": list(search_cfg["families"]),
        "axes": {key: search_cfg[key] for key in SEARCH_AXES},
        "spaces": search_cfg["spaces"],
        "model_bounds": MODEL_BOUNDS,
        "n_estimators": N_ESTIMATORS,
        "feature_code_version": FEATURE_CODE_VERSION,
        "sampler_seed": int(search_cfg["sampler_seed"]) + 1000 * int(fold["fold_index"]),
        "family_seed": seed_for(fold["fold_id"], family),
        "protocol_revision": PROTOCOL["revision"],
        "data_identity_hash": DATA_IDENTITY_HASH,
        "endpoint_hash": ENDPOINT_HASH,
    }
    return sha256_json(payload), payload


def search_study_path_and_id(label, fold, family):
    signature_hash, _ = search_signature(label, fold, family)
    study_id = f"rul-{signature_hash[:12]}"
    return STUDIES_DIR / f"{study_id}.sqlite3", study_id, signature_hash


def _load_or_create_study(label, fold, family):
    import optuna
    signature_hash, signature_payload = search_signature(label, fold, family)
    storage_path, study_id, _ = search_study_path_and_id(label, fold, family)
    budget = int(CONFIG["search"]["n_trials"])
    sampler_seed = int(signature_payload["sampler_seed"])
    storage = "sqlite:///" + str(storage_path)
    sampler = optuna.samplers.TPESampler(
        seed=sampler_seed, n_startup_trials=min(5, budget)
    )
    if storage_path.exists():
        study = optuna.load_study(study_name=study_id, storage=storage,
                                  sampler=sampler)
        if study.user_attrs.get("signature_hash") != signature_hash:
            raise RuntimeError("Estudo existente com assinatura incompatível.")
        if int(study.user_attrs.get("budget", -1)) != budget:
            raise RuntimeError("Estudo existente com orçamento incompatível.")
        if study.user_attrs.get("data_identity_hash") != DATA_IDENTITY_HASH:
            raise RuntimeError("Estudo existente pertence a outros dados.")
        if study.user_attrs.get("protocol_revision") != PROTOCOL["revision"]:
            raise RuntimeError("Estudo existente pertence a outro protocolo.")
    else:
        study = optuna.create_study(
            study_name=study_id,
            storage=storage,
            direction="minimize",
            sampler=sampler,
        )
        study.set_user_attr("signature_hash", signature_hash)
        study.set_user_attr("signature_payload", signature_payload)
        study.set_user_attr("budget", budget)
        study.set_user_attr("data_identity_hash", DATA_IDENTITY_HASH)
        study.set_user_attr("protocol_revision", PROTOCOL["revision"])
        study.set_user_attr("family", family)
        study.set_user_attr("label", label)
        study.set_user_attr("fold_id", fold["fold_id"])
    if len(study.trials) > budget:
        raise RuntimeError("Estudo com mais tentativas que o orçamento declarado.")
    return study, study_id, signature_hash


def search_remaining_trials(label, fold, family):
    """Consulta sem criar/treinar; usada no preview de ajustes."""
    storage_path, study_id, signature_hash = search_study_path_and_id(
        label, fold, family)
    if not storage_path.exists():
        return int(CONFIG["search"]["n_trials"]), 0
    import optuna
    study = optuna.load_study(study_name=study_id,
                              storage="sqlite:///" + str(storage_path))
    if study.user_attrs.get("signature_hash") != signature_hash:
        raise RuntimeError("Estudo existente com assinatura incompatível.")
    existing = len(study.trials)
    budget = int(CONFIG["search"]["n_trials"])
    if existing > budget:
        raise RuntimeError("Estudo com mais tentativas que o orçamento.")
    return budget - existing, existing


def _suggest_search_candidate(trial, family, fold):
    search_cfg = CONFIG["search"]
    spaces = search_cfg["spaces"]
    bin_hours = trial.suggest_categorical(
        "bin_hours", list(search_cfg["bin_hours_levels"]))
    representation = trial.suggest_categorical(
        "representation", list(search_cfg["representation_levels"]))
    preprocessing = trial.suggest_categorical(
        "preprocessing", list(search_cfg["preprocessing_levels"]))
    timing = bool(trial.suggest_categorical("timing", list(search_cfg["timing_levels"])))
    weights = trial.suggest_categorical(
        "weights", list(search_cfg["weights_levels"]))
    if family == "ridge":
        params = {"alpha": trial.suggest_categorical("alpha", spaces["ridge"]["alpha"])}
    elif family == "svr":
        params = {
            "C": trial.suggest_categorical("C", spaces["svr"]["C"]),
            "epsilon": trial.suggest_categorical("epsilon", spaces["svr"]["epsilon"]),
            "gamma": trial.suggest_categorical("gamma", spaces["svr"]["gamma"]),
        }
    elif family == "extra_trees":
        params = {
            "max_depth": trial.suggest_categorical(
                "max_depth", spaces["extra_trees"]["max_depth"]),
            "min_samples_leaf": trial.suggest_categorical(
                "min_samples_leaf", spaces["extra_trees"]["min_samples_leaf"]),
            "max_features": trial.suggest_categorical(
                "max_features", spaces["extra_trees"]["max_features"]),
        }
    else:
        raise ValueError(f"Família de busca inválida: {family}")
    params = validate_model_params(family, params)
    candidate = {
        "family": family,
        "params": params,
        "bin_hours": int(bin_hours),
        "representation": representation,
        "preprocessing": preprocessing,
        "timing": timing,
        "weights": weights,
        "seed": seed_for(fold["fold_id"], family),
        "origin": f"search:{family}:trial-{trial.number}",
        "fold_id": fold["fold_id"],
    }
    candidate["candidate_id"] = candidate_id(candidate)
    trial.set_user_attr("candidate", candidate)
    trial.set_user_attr("seed", int(candidate["seed"]))
    return candidate


def run_search_study(label, fold, family, target, get_features, masks, counter):
    study, study_id, signature_hash = _load_or_create_study(label, fold, family)
    budget = int(CONFIG["search"]["n_trials"])
    trials_before = len(study.trials)
    remaining = budget - trials_before
    meta = PROTOCOL["labels"][label]
    target_train = target.loc[masks["inner_train"]]
    target_val = target.loc[masks["val"]]

    def objective(trial):
        candidate = _suggest_search_candidate(trial, family, fold)
        features = get_features(candidate["bin_hours"])
        columns = feature_columns(candidate["bin_hours"],
                                  candidate["representation"],
                                  candidate["timing"])
        x_train = features.loc[target_train["endpoint_id"], columns]
        x_val = features.loc[target_val["endpoint_id"], columns]
        prediction_scaled, _, fit_metadata = fit_and_predict(
            candidate,
            x_train,
            target_train["y_scaled"].to_numpy(float),
            x_val,
            target_train["Falha"].to_numpy(int),
            target_train["phase"].to_numpy(str),
            counter=counter,
            purpose="selection",
        )
        prediction_metric = prediction_scaled * float(meta["divisor"]) * float(meta["metric_factor"])
        score, component_mae = selection_objective(
            target_val["y_metric"].to_numpy(float),
            prediction_metric,
            target_val["Falha"].to_numpy(int),
        )
        trial.set_user_attr("selection_macro_component_mae", float(score))
        trial.set_user_attr("component_mae", component_mae)
        trial.set_user_attr("n_train", int(len(target_train)))
        trial.set_user_attr("n_val", int(len(target_val)))
        trial.set_user_attr("fit_metadata", fit_metadata)
        return float(score)

    if remaining > 0:
        print(f"Optuna {label}/{fold['fold_id']}/{family}: "
              f"{trials_before} tentativas existentes, {remaining} novas, "
              f"orçamento total={budget}")
        study.optimize(objective, n_trials=remaining)
    else:
        print(f"Optuna {label}/{fold['fold_id']}/{family}: orçamento já completo "
              f"({trials_before}/{budget}); nenhuma tentativa nova.")

    complete = [trial for trial in study.trials
                if trial.state.name == "COMPLETE" and trial.value is not None]
    if not complete:
        raise RuntimeError(f"Estudo sem tentativa concluída: {study_id}")
    try:
        best = study.best_trial
    except Exception as exc:
        raise RuntimeError(f"Estudo sem melhor tentativa: {study_id}") from exc
    if "candidate" not in best.user_attrs:
        raise RuntimeError("Melhor tentativa sem candidato; estudo incompatível.")
    best_candidate = dict(best.user_attrs["candidate"])
    if best_candidate["family"] != family:
        raise RuntimeError("Melhor tentativa pertence a outra família.")
    trial_rows = []
    for trial in study.trials:
        trial_rows.append({
            "label": label,
            "fold_id": fold["fold_id"],
            "family": family,
            "study_id": study_id,
            "trial_number": int(trial.number),
            "state": trial.state.name,
            "value": None if trial.value is None else float(trial.value),
            "candidate": trial.user_attrs.get("candidate"),
            "seed": trial.user_attrs.get("seed"),
            "component_mae": trial.user_attrs.get("component_mae"),
            "n_train": trial.user_attrs.get("n_train"),
            "n_val": trial.user_attrs.get("n_val"),
        })
    return {
        "family": family,
        "study_id": study_id,
        "signature_hash": signature_hash,
        "storage": str(search_study_path_and_id(label, fold, family)[0]),
        "budget_total": budget,
        "trials_before": int(trials_before),
        "trials_after": int(len(study.trials)),
        "new_trials": int(len(study.trials) - trials_before),
        "best_trial": int(best.number),
        "best_score": float(best.value),
        "candidate": best_candidate,
        "trials": trial_rows,
    }


print("Funções de busca prontas (opcional; use CONFIG['search']['enabled']=True).")


In [ ]:
# =============================================================================
# SEÇÃO 12 — SELEÇÃO INTERNA POR FAMÍLIA, REFIT E TESTE EXTERNO
# Cada família é um resultado independente. Não há vencedor entre famílias.
# =============================================================================


def masks_for_fold(target, fold):
    return _cer_rul.masks_for_fold(
        target, fold,
        id_to_equipment=ID_TO_EQUIPMENT,
        equipment_order=EQUIPMENT_ORDER,
    )


def evaluate_candidate_inner(candidate, target, masks, get_features, counter):
    meta = PROTOCOL["labels"][target["label"].iloc[0]]
    train = target.loc[masks["inner_train"]]
    validation = target.loc[masks["val"]]
    if candidate["family"] == "mean":
        x_train = pd.DataFrame(index=pd.Index(train["endpoint_id"]))
        x_val = pd.DataFrame(index=pd.Index(validation["endpoint_id"]))
    else:
        features = get_features(candidate["bin_hours"])
        columns = feature_columns(candidate["bin_hours"],
                                  candidate["representation"], candidate["timing"])
        x_train = features.loc[train["endpoint_id"], columns]
        x_val = features.loc[validation["endpoint_id"], columns]
    prediction_scaled, _, fit_metadata = fit_and_predict(
        candidate,
        x_train,
        train["y_scaled"].to_numpy(float),
        x_val,
        train["Falha"].to_numpy(int),
        train["phase"].to_numpy(str),
        counter=counter,
        purpose="selection",
    )
    prediction_metric = (prediction_scaled * float(meta["divisor"])
                         * float(meta["metric_factor"]))
    score, component_mae = selection_objective(
        validation["y_metric"].to_numpy(float),
        prediction_metric,
        validation["Falha"].to_numpy(int),
    )
    return {
        "candidate": candidate,
        "score": float(score),
        "component_mae": component_mae,
        "n_train": int(len(train)),
        "n_val": int(len(validation)),
        "fit_metadata": fit_metadata,
        "selection_source": "inner_fit",
    }


def select_family_inner(label, fold, family, target, masks, get_features, counter):
    """Seleciona NO MÁXIMO uma configuração daquela família, nunca entre famílias."""
    mode = "search" if CONFIG["search"]["enabled"] else "quick"
    if family not in MODEL_FAMILIES:
        raise ValueError(f"Família desconhecida: {family}")
    search_results = []
    search_trials = []
    if family == "mean":
        candidate = candidate_from_preset("mean", {}, fold["fold_id"], 0)
        records = [evaluate_candidate_inner(candidate, target, masks,
                                            get_features, counter)]
    elif mode == "quick":
        candidates = [candidate for candidate in quick_candidates(fold["fold_id"])
                      if candidate["family"] == family]
        if not candidates:
            raise RuntimeError(f"Sem candidatos quick para {family}.")
        records = [evaluate_candidate_inner(candidate, target, masks,
                                           get_features, counter)
                   for candidate in candidates]
    else:
        result = run_search_study(label, fold, family, target,
                                  get_features, masks, counter)
        best_trial_row = next(
            row for row in result["trials"]
            if int(row["trial_number"]) == int(result["best_trial"])
        )
        search_results.append({key: result[key] for key in
                               ("family", "study_id", "budget_total",
                                "trials_before", "trials_after", "new_trials",
                                "best_trial", "best_score", "candidate")})
        search_trials.extend(result["trials"])
        records = [{
            "candidate": result["candidate"],
            "score": float(result["best_score"]),
            "component_mae": best_trial_row.get("component_mae"),
            "n_train": int(masks["inner_train"].sum()),
            "n_val": int(masks["val"].sum()),
            "fit_metadata": None,
            "selection_source": f"optuna_best/{result['study_id']}",
        }]
    if not records:
        raise RuntimeError(f"Nenhum candidato interno para {family}.")
    chosen_record = min(
        records,
        key=lambda record: (float(record["score"]),
                            record["candidate"]["candidate_id"]),
    )
    chosen = chosen_record["candidate"]
    if chosen["family"] != family:
        raise AssertionError("A seleção interna misturou famílias.")
    if family == "mean" and chosen["weights"] != "none":
        raise AssertionError("A média de referência precisa ser sem pesos.")
    return {
        "mode": mode,
        "family": family,
        "chosen": chosen,
        "chosen_record": chosen_record,
        "records": records,
        "search_results": search_results,
        "search_trials": search_trials,
    }


def _selected_model_predict(candidate, target, masks, get_features, counter,
                            purpose="final"):
    train = target.loc[masks["outer_train"]]
    test = target.loc[masks["test"]]
    if candidate["family"] == "mean":
        x_train = pd.DataFrame(index=pd.Index(train["endpoint_id"]))
        x_test = pd.DataFrame(index=pd.Index(test["endpoint_id"]))
    else:
        features = get_features(candidate["bin_hours"])
        columns = feature_columns(candidate["bin_hours"],
                                  candidate["representation"], candidate["timing"])
        x_train = features.loc[train["endpoint_id"], columns]
        x_test = features.loc[test["endpoint_id"], columns]
    prediction_scaled, fitted, fit_metadata = fit_and_predict(
        candidate,
        x_train,
        train["y_scaled"].to_numpy(float),
        x_test,
        train["Falha"].to_numpy(int),
        train["phase"].to_numpy(str),
        counter=counter,
        purpose=purpose,
    )
    return test, prediction_scaled, fitted, fit_metadata


def execute_label_fold_family(label, fold, family, get_features, counter):
    target = TARGETS[label]
    masks = masks_for_fold(target, fold)
    expected_keys = target.loc[masks["test"], "endpoint_id"].tolist()
    candidate_mode = "search" if CONFIG["search"]["enabled"] else "quick"
    identity = run_identity_for(label, fold, family, candidate_mode,
                                expected_keys)
    path = run_file_path(label, fold["fold_id"], family)
    existing = load_complete_run(path, identity, expected_keys)
    if existing is not None:
        print(f"{label}/{fold['fold_id']}/{family}: execução completa reutilizada.")
        return existing, True

    counter_before = dict(counter)
    selection = select_family_inner(label, fold, family, target, masks,
                                    get_features, counter)
    chosen = selection["chosen"]
    test, prediction_scaled, fitted, fit_metadata = _selected_model_predict(
        chosen, target, masks, get_features, counter
    )
    prediction_frame = predictions_frame(test, prediction_scaled, label, fold, chosen)
    if len(prediction_frame) != len(expected_keys):
        raise RuntimeError("Predições ausentes antes de gravar o JSON.")
    meta = PROTOCOL["labels"][label]
    metrics = metric_breakdown(
        prediction_frame["y_metric"].to_numpy(float),
        prediction_frame["pred_metric"].to_numpy(float),
        prediction_frame["Falha"].to_numpy(int),
        prediction_frame["equipment"].to_numpy(str),
        prediction_frame["phase"].to_numpy(str),
        meta["unit"],
    )
    delta = {key: int(counter.get(key, 0) - counter_before.get(key, 0))
             for key in sorted(counter)}
    payload = {
        "run_identity": identity,
        "run_id": RUN_ID,
        "label": label,
        "family": family,
        "unit": meta["unit"],
        "divisor": float(meta["divisor"]),
        "metric_factor": float(meta["metric_factor"]),
        "fold_id": fold["fold_id"],
        "fold": {key: (list(value) if isinstance(value, tuple) else value)
                 for key, value in fold.items()},
        "candidate_mode": selection["mode"],
        "chosen_candidate": chosen,
        "chosen_record": {key: value for key, value in selection["chosen_record"].items()
                          if key != "candidate"},
        "selection_records": selection["records"],
        "search_results": selection["search_results"],
        "search_trials": selection["search_trials"],
        "fit_metadata": fit_metadata,
        "fit_counts_this_run": delta,
        "counts": {
            "n_endpoints": int(len(target)),
            "n_inner_train": int(masks["inner_train"].sum()),
            "n_val": int(masks["val"].sum()),
            "n_outer_train": int(masks["outer_train"].sum()),
            "n_outer_test": int(masks["test"].sum()),
        },
        "metrics": metrics,
        "predictions": predictions_to_json(prediction_frame),
    }
    save_complete_run(path, payload)
    print(f"{label}/{fold['fold_id']}/{family}: escolhido "
          f"{chosen['candidate_id']} ({chosen['family']}), "
          f"teste={len(prediction_frame)}, "
          f"MAE={metrics['overall']['mae']:.4f} {meta['unit']}")
    return payload, False


def expected_run_count():
    return int(len(LABEL_ORDER) * len(SPLITS) * len(MODEL_FAMILIES))


def completed_run_payload(label, fold, family):
    """Leitura estrita: JSON incompleto/adulterado levanta erro, não é pulado."""
    target = TARGETS[label]
    masks = masks_for_fold(target, fold)
    expected_keys = target.loc[masks["test"], "endpoint_id"].tolist()
    candidate_mode = "search" if CONFIG["search"]["enabled"] else "quick"
    identity = run_identity_for(label, fold, family, candidate_mode,
                                expected_keys)
    path = run_file_path(label, fold["fold_id"], family)
    if not path.exists():
        return None
    return load_complete_run(path, identity, expected_keys)


def estimate_expected_fits():
    mode = "search" if CONFIG["search"]["enabled"] else "quick"
    mean_selection = 0
    mean_final = 0
    learned_selection = 0
    final_refits = 0
    for label in LABEL_ORDER:
        for fold in SPLITS.to_dict(orient="records"):
            for family in MODEL_FAMILIES:
                if completed_run_payload(label, fold, family) is not None:
                    continue
                final_refits += 1
                if family == "mean":
                    mean_selection += 1
                    mean_final += 1
                elif mode == "quick":
                    learned_selection += len([
                        candidate for candidate in quick_candidates(fold["fold_id"])
                        if candidate["family"] == family
                    ])
                else:
                    remaining, _ = search_remaining_trials(label, fold, family)
                    learned_selection += remaining
    ablation_fits = 0
    if CONFIG["ablation"]["enabled"]:
        ablation_fits = len(CONFIG["ablation"]["levels"]) * len(LABEL_ORDER) * len(SPLITS)
    total = mean_selection + learned_selection + final_refits + ablation_fits
    return {
        "mode": mode,
        "expected_runs": expected_run_count(),
        "mean_selection_fits": int(mean_selection),
        "mean_final_fits": int(mean_final),
        "baseline_fits_total": int(mean_selection + mean_final),
        "learned_selection_fits": int(learned_selection),
        "final_refits": int(final_refits),
        "ablation_fits": int(ablation_fits),
        "total_model_fits": int(total),
    }


def print_fit_preview(preview):
    print("\nPREVIEW DE AJUSTES (antes de qualquer treino desta execução)")
    print("  modo:", preview["mode"])
    print("  execuções label×fold×família esperadas:", preview["expected_runs"])
    print("  média (seleção interna):", preview["mean_selection_fits"])
    print("  média (refit final):", preview["mean_final_fits"])
    print("  baseline total (seleção + final):", preview["baseline_fits_total"])
    print("  candidatos aprendidos (seleção interna):",
          preview["learned_selection_fits"])
    print("  refits finais de todas as famílias:", preview["final_refits"])
    print("  ablação:", preview["ablation_fits"])
    print("  TOTAL de ajustes esperados:", preview["total_model_fits"])
    print("  JSON completo no disco é reutilizado e não conta.")


print("Orquestração por família pronta.")


In [ ]:
# =============================================================================
# SEÇÃO 13 — AGREGAÇÃO, RELATÓRIOS, GRÁFICOS E PACOTE DE RESULTADOS
# =============================================================================


def write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def export_data_tables():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    write_csv(AUDIT_TABLE, RUN_DIR / "cleanup_audit.csv")
    atomic_write_json(RUN_DIR / "cleanup_audit_full.json", AUDITS)
    write_csv(SENSOR_EDA, RUN_DIR / "eda_sensor_quantiles.csv")
    write_csv(TIME_GAP_EDA, RUN_DIR / "eda_time_gaps.csv")
    write_csv(TERMINAL_TABLE, RUN_DIR / "terminal_targets.csv")
    endpoint_export = ENDPOINTS.copy()
    endpoint_export["Date"] = endpoint_export["Date"].map(
        lambda value: pd.Timestamp(value).isoformat())
    write_csv(endpoint_export, RUN_DIR / "endpoint_index.csv")
    split_export = SPLITS.copy()
    for column in ("test_ids", "val_ids", "inner_train_ids", "outer_train_ids"):
        split_export[column] = split_export[column].map(
            lambda values: ",".join(str(value) for value in values))
    for column in ("inner_train_equipment",):
        split_export[column] = split_export[column].map(
            lambda values: ",".join(str(value) for value in values))
    write_csv(split_export, RUN_DIR / "splits.csv")
    atomic_write_json(RUN_DIR / "data_identity.json", DATA_IDENTITY)


def predictions_from_payloads(payloads):
    rows = []
    for payload in payloads:
        for record in payload["predictions"]:
            rows.append(record)
    if not rows:
        return pd.DataFrame(columns=[
            "label", "unit", "fold_id", "test_equipment", "endpoint_id",
            "Falha", "equipment", "Date", "phase", "candidate_id", "family",
            "y_native", "pred_native", "y_metric", "pred_metric",
            "residual_metric", "split_role",
        ])
    frame = pd.DataFrame(rows)
    frame["Date"] = pd.to_datetime(frame["Date"])
    return frame


def chosen_config_rows(payloads):
    rows = []
    for payload in payloads:
        candidate = payload["chosen_candidate"]
        rows.append({
            "label": payload["label"],
            "fold_id": payload["fold_id"],
            "test_equipment": payload["fold"]["test_equipment"],
            "candidate_mode": payload["candidate_mode"],
            "candidate_id": candidate["candidate_id"],
            "family": payload["family"],
            "bin_hours": candidate["bin_hours"],
            "representation": candidate["representation"],
            "preprocessing": candidate["preprocessing"],
            "timing": candidate["timing"],
            "weights": candidate["weights"],
            "seed": candidate["seed"],
            "params": canonical_json(candidate["params"]),
            "selection_score": payload["chosen_record"]["score"],
            "inner_component_mae": canonical_json(
                payload["chosen_record"].get("component_mae")),
        })
    return pd.DataFrame(rows)


def trials_rows(payloads):
    rows = []
    for payload in payloads:
        rows.extend(payload.get("search_trials", []))
    if not rows:
        return pd.DataFrame()
    result = pd.DataFrame(rows)
    result["candidate"] = result["candidate"].map(canonical_json)
    result["component_mae"] = result["component_mae"].map(
        lambda value: canonical_json(value) if value is not None else "")
    return result


def plot_predictions(prediction_frame):
    if prediction_frame.empty:
        return []
    outputs = []
    for (label, family), label_frame in prediction_frame.groupby(
            ["label", "family"], sort=False):
        folds = sorted(label_frame["fold_id"].unique())
        metric_y, metric_pred, unit = plot_metric_columns(label)
        figure, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=False)
        axes = np.ravel(axes)
        for axis, fold_id in zip(axes, folds):
            fold_frame = label_frame[label_frame["fold_id"] == fold_id]
            for fault_id, component in fold_frame.groupby("Falha", sort=True):
                component = component.sort_values("Date")
                axis.plot(component["Date"], component[metric_y],
                          color="black", linewidth=1.0, alpha=0.55)
                axis.plot(component["Date"], component[metric_pred],
                          linewidth=1.2, label=f"Falha {int(fault_id)}")
            axis.set_title(f"{label} | {family} | {fold_id} | teste "
                           f"{fold_frame['test_equipment'].iloc[0]}")
            axis.set_ylabel(unit)
            axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            axis.tick_params(axis="x", rotation=30)
            axis.grid(alpha=0.25)
            axis.legend(fontsize=7, ncol=2)
        for axis in axes[len(folds):]:
            axis.axis("off")
        figure.suptitle(f"Predições em datas reais — {label} / {family} (sem clamp)")
        figure.tight_layout()
        path = PLOTS_DIR / f"predicoes_{label}__{family}.png"
        figure.savefig(path, dpi=140)
        plt.close(figure)
        outputs.append(path)
    return outputs


def artifact_hashes(root):
    artifacts = {}
    for path in sorted(Path(root).rglob("*")):
        if not path.is_file():
            continue
        if path.name == "manifest.json":
            continue
        relative = path.relative_to(root).as_posix()
        artifacts[relative] = sha256_file(path)
    return artifacts


def build_manifest(payloads, preview, fit_counter, started_utc, finished_utc,
                   metric_macro, ablation_results):
    chosen = chosen_config_rows(payloads)
    trial_table = trials_rows(payloads)
    manifest = {
        "status": ("training_complete" if CONFIG["run_training"]
                   and len(payloads) == expected_run_count()
                   else "smoke_without_training" if not CONFIG["run_training"]
                   else "incomplete"),
        "run_id": RUN_ID,
        "started_utc": started_utc,
        "finished_utc": finished_utc,
        "source": SOURCE_PROVENANCE,
        "protocol": PROTOCOL,
        "versions": RUNTIME_VERSIONS,
        "config": CONFIG,
        "effective_config": EFFECTIVE_CONFIG,
        "effective_config_hash": EFFECTIVE_CONFIG_HASH,
        "protocol_hash": PROTOCOL_HASH,
        "data_identity": DATA_IDENTITY,
        "data_identity_hash": DATA_IDENTITY_HASH,
        "data": {
            "input_files": [
                {"label": label, "filename": INPUT_FILES[label].name,
                 "sha256": RAW_FILE_HASHES[label]}
                for label in LABEL_ORDER
            ],
            "clean_frame_sha256": CLEAN_HASHES,
            "cleanup_audit": AUDIT_TABLE.to_dict(orient="records"),
            "cleanup_audit_full": "cleanup_audit_full.json",
            "eda_sensor_quantiles": "eda_sensor_quantiles.csv",
            "eda_time_gaps": "eda_time_gaps.csv",
            "terminal_targets": TERMINAL_TABLE.assign(
                Date=TERMINAL_TABLE["Date"].map(lambda value: pd.Timestamp(value).isoformat())
            ).to_dict(orient="records"),
            "counts": {
                label: {
                    "raw_rows": AUDITS[label]["raw_rows"],
                    "clean_keys": AUDITS[label]["clean_keys"],
                    "target_conflict_keys": AUDITS[label]["target_conflict_keys"],
                    "terminal_nonzero_count": AUDITS[label]["terminal_nonzero_count"],
                }
                for label in LABEL_ORDER
            },
        },
        "endpoints": {
            "sha256": ENDPOINT_HASH,
            "count": int(len(ENDPOINTS)),
            "per_fault": ENDPOINT_COUNTS,
            "warmup_records": PROTOCOL["warmup_records"],
            "stride": PROTOCOL["endpoint_stride"],
            "coverage": "post_warmup_only_not_all_original_rows",
        },
        "splits": {
            "sha256": SPLIT_HASH,
            "folds": SPLITS.to_dict(orient="records"),
        },
        "training": {
            "run_training": bool(CONFIG["run_training"]),
            "expected_runs": expected_run_count(),
            "observed_runs": int(len(payloads)),
            "counts_by_family": {
                family: int(sum(payload["family"] == family for payload in payloads))
                for family in MODEL_FAMILIES
            },
            "preview_fit_counts": preview,
            "actual_fit_counts": dict(fit_counter),
            "chosen_configs": chosen.to_dict(orient="records"),
            "search_trials": trial_table.to_dict(orient="records"),
            "metric_macro": metric_macro.to_dict(orient="records")
            if isinstance(metric_macro, pd.DataFrame) and not metric_macro.empty else [],
            "ablation": ablation_results,
            "notes": [
                "uma configuração escolhida por família, exclusivamente pela MAE macro por componente no fold interno",
                "média sempre refeita/testada como família independente; não é apenas candidata interna",
                "teste externo intocado até o refit final de cada família",
                "nenhuma família é escolhida como vencedora pelo teste externo",
                "sem clamp, sem MAPE, sem ranqueamento entre rótulos",
                "métricas por fold usam o equipamento de teste; macro entre folds é média simples por família",
                "predições por família/fold estão embutidas nos JSON de execução",
            ],
        },
        "artifacts": {},
    }
    return manifest


def export_results(payloads, preview, fit_counter, started_utc, finished_utc,
                   ablation_results=None):
    if ablation_results is None:
        ablation_results = {}
    export_data_tables()
    predictions = predictions_from_payloads(payloads)
    metric_rows = fold_metric_rows(predictions) if not predictions.empty else []
    metrics_folds = pd.DataFrame(metric_rows)
    metrics_macro = aggregate_metric_rows(metric_rows)
    if not metrics_folds.empty:
        write_csv(metrics_folds, RUN_DIR / "metrics_folds.csv")
    if not metrics_macro.empty:
        write_csv(metrics_macro, RUN_DIR / "metrics_macro.csv")
        write_csv(metrics_macro[metrics_macro["scope"] == "equipment_macro"],
                  RUN_DIR / "equipment_macro.csv")
    write_csv(predictions, RUN_DIR / "predictions_all.csv")
    chosen = chosen_config_rows(payloads)
    write_csv(chosen, RUN_DIR / "chosen_configs.csv")
    trial_table = trials_rows(payloads)
    if not trial_table.empty:
        write_csv(trial_table, RUN_DIR / "search_trials.csv")
    fit_rows = []
    for payload in payloads:
        fit_rows.append({
            "label": payload["label"],
            "family": payload["family"],
            "fold_id": payload["fold_id"],
            "candidate_mode": payload["candidate_mode"],
            "candidate_id": payload["chosen_candidate"]["candidate_id"],
            "model_fits": payload["fit_counts_this_run"].get("model_fits", 0),
            "baseline_selection_fits": payload["fit_counts_this_run"].get(
                "selection_baseline_fits", 0),
            "baseline_final_fits": payload["fit_counts_this_run"].get(
                "final_baseline_fits", 0),
        })
    write_csv(pd.DataFrame(fit_rows), RUN_DIR / "fit_counts_by_run.csv")
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    plot_paths = plot_predictions(predictions)
    manifest = build_manifest(payloads, preview, fit_counter, started_utc,
                              finished_utc, metrics_macro, ablation_results)
    manifest["artifacts"] = artifact_hashes(RUN_DIR)
    atomic_write_json(RUN_DIR / "manifest.json", manifest)

    zip_path = EXPORTS_DIR / f"{RUN_ID}.zip"
    temporary = zip_path.with_name(f".{zip_path.name}.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=(Path(RUN_ID) /
                                             path.relative_to(RUN_DIR)).as_posix())
    os.replace(temporary, zip_path)
    print("ZIP de resultados:", zip_path)
    print("CACHE e arquivos de entrada NÃO foram incluídos no ZIP.")
    if CONFIG["output"]["download_zip"] and IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    return {
        "predictions": predictions,
        "metrics_folds": metrics_folds,
        "metrics_macro": metrics_macro,
        "chosen": chosen,
        "trials": trial_table,
        "manifest": manifest,
        "zip_path": zip_path,
        "plots": plot_paths,
    }


print("Relatórios e exportação prontos.")


## Ablação: um eixo, todos os níveis, sem vencedor externo

Quando habilitada, a ablação fixa família, parâmetros, largura, seed e os demais eixos, alterando **apenas** `representation`, `timing` ou `weights`. Cada nível pré-declarado é treinado no mesmo treino externo e avaliado no mesmo teste externo. Todos são reportados; **não** há escolha do “melhor nível” pelo teste, nem ranqueamento entre níveis. O resultado serve para descrever sensibilidade, não para selecionar o modelo principal.


In [ ]:
# =============================================================================
# SEÇÃO 14 — ABLAÇÃO OPCIONAL DE UM ÚNICO EIXO
# =============================================================================

ABLATION_AXES = {
    "representation": ("representation", list(REPRESENTATIONS)),
    "timing": ("timing", [False, True]),
    "weights": ("weights", list(WEIGHT_SCHEMES)),
}


def ablation_base_candidate(fold_id):
    cfg = CONFIG["ablation"]
    if cfg["base_family"] not in ("ridge", "svr", "extra_trees"):
        raise ValueError("Ablação exige uma família aprendida.")
    candidate = {
        "family": cfg["base_family"],
        "params": validate_model_params(cfg["base_family"], cfg["base_params"]),
        "bin_hours": int(cfg["base_bin_hours"]),
        "representation": cfg["base_representation"],
        "preprocessing": cfg["base_preprocessing"],
        "timing": bool(cfg["base_timing"]),
        "weights": cfg["base_weights"],
        "seed": seed_for(fold_id, cfg["base_family"]),
        "origin": "ablation",
        "fold_id": fold_id,
    }
    if candidate["bin_hours"] not in PROTOCOL["bin_hours_candidates"]:
        raise ValueError("Ablação: bin_hours inválido.")
    if candidate["representation"] not in REPRESENTATIONS:
        raise ValueError("Ablação: representação inválida.")
    if candidate["preprocessing"] not in PREPROCESSING_OPTIONS:
        raise ValueError("Ablação: pré-processamento inválido.")
    if candidate["weights"] not in WEIGHT_SCHEMES:
        raise ValueError("Ablação: pesos inválidos.")
    candidate["candidate_id"] = candidate_id(candidate)
    return candidate


def ablation_levels():
    cfg = CONFIG["ablation"]
    if cfg["axis"] not in ABLATION_AXES:
        raise ValueError("Ablação: eixo deve ser representation, timing ou weights.")
    key, valid = ABLATION_AXES[cfg["axis"]]
    levels = list(cfg["levels"])
    if not levels or len(levels) != len(set(map(str, levels))):
        raise ValueError("Ablação: níveis vazios ou duplicados.")
    for level in levels:
        if level not in valid:
            raise ValueError(f"Ablação: nível inválido para {cfg['axis']}: {level!r}")
    return key, levels


def ablation_candidate_for_level(base, key, level, fold_id):
    candidate = json.loads(canonical_json(base))
    candidate[key] = level
    candidate["seed"] = seed_for(fold_id, base["family"])
    candidate["origin"] = f"ablation:{key}:{level}"
    candidate["candidate_id"] = candidate_id(candidate)
    return candidate


def run_ablation(get_features, counter):
    cfg = CONFIG["ablation"]
    if not cfg["enabled"]:
        return {}
    key, levels = ablation_levels()
    rows = []
    prediction_frames = []
    metric_rows = []
    for label in LABEL_ORDER:
        target = TARGETS[label]
        meta = PROTOCOL["labels"][label]
        for fold in SPLITS.to_dict(orient="records"):
            masks = masks_for_fold(target, fold)
            base = ablation_base_candidate(fold["fold_id"])
            for level in levels:
                candidate = ablation_candidate_for_level(
                    base, key, level, fold["fold_id"])
                test, prediction_scaled, _, fit_metadata = _selected_model_predict(
                    candidate, target, masks, get_features, counter,
                    purpose="ablation"
                )
                frame = predictions_frame(test, prediction_scaled, label,
                                          fold, candidate)
                frame["ablation_axis"] = key
                frame["ablation_level"] = str(level)
                prediction_frames.append(frame)
                block = metric_breakdown(
                    frame["y_metric"].to_numpy(float),
                    frame["pred_metric"].to_numpy(float),
                    frame["Falha"].to_numpy(int),
                    frame["equipment"].to_numpy(str),
                    frame["phase"].to_numpy(str),
                    meta["unit"],
                )
                for scope, values in (("overall", {"all": block["overall"]}),
                                      ("component", block["by_component"]),
                                      ("phase", block["by_phase"])):
                    for name, metrics in values.items():
                        metric_rows.append({
                            "label": label,
                            "family": base["family"],
                            "unit": meta["unit"],
                            "fold_id": fold["fold_id"],
                            "axis": key,
                            "level": str(level),
                            "scope": scope,
                            "name": name,
                            **metrics,
                            "aggregation": "ablation_no_winner",
                        })
                rows.append({
                    "label": label,
                    "fold_id": fold["fold_id"],
                    "axis": key,
                    "level": str(level),
                    "candidate_id": candidate["candidate_id"],
                    "family": candidate["family"],
                    "n_functional": fit_metadata.get("n_features_out", 0),
                })
    predictions = pd.concat(prediction_frames, ignore_index=True)
    metrics = pd.DataFrame(metric_rows)
    if predictions.empty or metrics.empty:
        raise RuntimeError("Ablação não produziu resultados.")
    ABLATION_DIR.mkdir(parents=True, exist_ok=True)
    write_csv(pd.DataFrame(rows), ABLATION_DIR / "ablation_fits.csv")
    write_csv(predictions, ABLATION_DIR / "ablation_predictions.csv")
    write_csv(metrics, ABLATION_DIR / "ablation_metrics.csv")
    summary = {
        "enabled": True,
        "family": cfg["base_family"],
        "axis": key,
        "levels": [str(level) for level in levels],
        "n_predictions": int(len(predictions)),
        "n_levels": int(len(levels)),
        "no_outer_winner_selection": True,
        "metrics": metrics.to_dict(orient="records"),
    }
    atomic_write_json(ABLATION_DIR / "ablation_summary.json", summary)
    print("Ablação concluída:", key, levels, "| sem seleção de vencedor.")
    display(metrics.groupby(["family", "axis", "level", "scope"], sort=False)
            [["mae", "rmse", "bias"]].mean().reset_index())
    return summary


print("Ablação pronta (desabilitada por padrão).")


## Testes leves de runtime

Esta seção não ajusta modelos. Os testes cobrem:

- média de duplicatas com multiplicidade e conflito de alvo (média de **todas** as linhas, não dos valores únicos);
- conflito de sensores na mesma chave, que deve falhar;
- alinhamento bruto/limpo dos cinco rótulos;
- inclusão obrigatória do endpoint final;
- bin `(left, right]`, ancoragem à direita e imutabilidade do contexto ao acrescentar uma linha futura;
- inclinação por dia real;
- splits disjuntos;
- imputador/scaler ajustados só no treino, incluindo coluna totalmente vazia;
- pesos normalizados;
- unidades nativas, inclusive predição negativa sem clamp;
- fingerprint determinístico e rejeição de JSON de retomada sem predições.


In [ ]:
# =============================================================================
# SEÇÃO 15 — TESTES LEVES (SEM AJUSTE PESADO)
# =============================================================================


def _synthetic_frame(n=40, fault_id=1):
    dates = pd.date_range("2024-01-01", periods=n, freq="h")
    frame = pd.DataFrame({
        "Date": dates,
        "Falha": [fault_id] * n,
        "RUL": np.linspace(80.0, 0.0, n),
        "raw_count": [1] * n,
    })
    for position, column in enumerate(SENSOR_COLUMNS):
        frame[column] = np.linspace(1.0 + position, 2.0 + position, n)
    return frame.loc[:, ["Date", "Falha", "RUL", "raw_count"] + list(SENSOR_COLUMNS)]


def run_self_tests():
    results = []

    def check(name, condition):
        if not condition:
            raise AssertionError(name)
        results.append(name)

    # 1. Consolidação: média de todas as linhas, inclusive repetidas.
    base_date = pd.Timestamp("2024-01-01 00:00:00")
    raw = pd.DataFrame({
        "Date": [base_date, base_date, base_date, base_date + pd.Timedelta(hours=1)],
        "Falha": [1, 1, 1, 1],
        "RUL": [1.0, 1.0, 0.0, 5.0],
    })
    for position, column in enumerate(SENSOR_COLUMNS):
        raw[column] = [1.0 + position, 1.0 + position, 1.0 + position,
                       1.01 + position]
    raw = raw.loc[:, list(CSV_COLUMNS)]
    clean, audit, _ = consolidate_frame(raw, "synthetic", enforce_expectations=False)
    first_target = float(clean.loc[clean["Date"] == base_date, "RUL"].iloc[0])
    check("duplicate_mean_all_rows", abs(first_target - (2.0 / 3.0)) < 1e-12)
    check("duplicate_mean_not_unique_values", abs(first_target - 0.5) > 1e-6)
    check("target_conflict_audit", audit["target_conflict_keys"] == 1)
    check("multiplicity_audit", audit["raw_max_multiplicity"] == 3)
    check("fault_ids_exact_integer",
          coerce_fault_ids(["1", "7"]).tolist() == [1, 7])
    try:
        coerce_fault_ids(["1", "1.9"])
        raise AssertionError("Falha fracionária deveria falhar")
    except ValueError:
        results.append("fault_ids_fractional_rejected")
    clean_fp = clean_frame_fingerprint(clean)
    check("clean_fingerprint_stable", clean_fp == clean_frame_fingerprint(clean))
    changed_clean = clean.copy()
    changed_clean.loc[0, "RUL"] = changed_clean.loc[0, "RUL"] + 0.01
    check("clean_fingerprint_sensitive",
          clean_fp != clean_frame_fingerprint(changed_clean))

    conflicting = raw.copy()
    conflicting.loc[1, SENSOR_COLUMNS[0]] = 999.0
    try:
        consolidate_frame(conflicting, "synthetic", enforce_expectations=False)
        raise AssertionError("feature conflict deveria falhar")
    except ValueError:
        results.append("feature_conflict_fails")

    # 2. Alinhamento bruto/limpo nos cinco rótulos (sintético).
    raw_map = {}
    clean_map = {}
    for label in LABEL_ORDER:
        frame = raw.copy()
        frame["RUL"] = frame["RUL"] + LABEL_ORDER.index(label)
        raw_map[label] = frame
        clean_map[label] = _synthetic_frame(n=4)
    check("raw_alignment", assert_raw_alignment(raw_map))
    check("clean_alignment", assert_clean_alignment(clean_map))

    # 3. Endpoint final sempre incluído, e nenhum dentro do aquecimento.
    endpoint_reference = _synthetic_frame(n=40)
    endpoints = build_endpoint_rows(endpoint_reference, warmup=5, stride=7,
                                    enforce=False)
    for fault_id, group in endpoints.groupby("Falha", sort=True):
        check(f"endpoint_first_after_warmup_{fault_id}", group["record_index"].min() >= 5)
        check(f"endpoint_final_{fault_id}", group["record_index"].max() == 39)
        check(f"endpoint_no_after_final_{fault_id}",
              group["n_records_after"].min() == 0)

    # 4. Bin (left, right] e ancoragem à direita.
    endpoint_date = pd.Timestamp("2024-01-03 12:00:00")
    sparse_dates = pd.Series([
        endpoint_date - pd.Timedelta(hours=13),
        endpoint_date - pd.Timedelta(hours=12),  # exatamente na borda: entra no bin anterior
        endpoint_date - pd.Timedelta(hours=11),
        endpoint_date,
    ])
    sparse_values = np.vstack([np.ones(8) * value for value in (10, 20, 30, 40)])
    last_values, last_times, counts = _bin_stats(
        sparse_dates, sparse_values, endpoint_date, 12)
    check("bin_left_boundary_excluded_from_right", counts[30] == 2 and counts[31] == 2)
    check("bin_last_value_causal", last_values[30, 0] == 20 and last_values[31, 0] == 40)
    check("bin_last_time_causal",
          last_times.iloc[30] == endpoint_date - pd.Timedelta(hours=12)
          and last_times.iloc[31] == endpoint_date)

    def same_features(first, second):
        return all(
            (math.isnan(first[key]) and math.isnan(second[key]))
            or abs(first[key] - second[key]) < 1e-12
            for key in first
        )

    # 5. Acrescentar uma linha futura não altera o contexto já construído.
    group = _synthetic_frame(n=80)
    endpoint_row = type("Endpoint", (), {
        "record_index": 60, "Date": group.loc[60, "Date"]
    })()
    before = build_feature_row(endpoint_row, group, 12)
    future = group.iloc[:61].copy()
    future_row = future.iloc[[-1]].copy()
    future_row["Date"] = future["Date"].max() + pd.Timedelta(hours=1)
    for column in SENSOR_COLUMNS:
        future_row[column] = future_row[column] + 1000.0
    future_row["Falha"] = 1
    future_row["RUL"] = 0.0
    future_row["raw_count"] = 1
    after = build_feature_row(endpoint_row, pd.concat([group, future_row],
                                                      ignore_index=True), 12)
    check("future_row_does_not_change_past_context", same_features(before, after))

    # 5b. Registros antigos fora da grade não influenciam valor/resumo da janela.
    long_group = _synthetic_frame(n=600)
    long_group.loc[:199, list(SENSOR_COLUMNS)] += 10000.0
    long_endpoint = type("Endpoint", (), {
        "record_index": 590, "Date": long_group.loc[590, "Date"]
    })()
    full_window = build_feature_row(long_endpoint, long_group, 12)
    trimmed = long_group.iloc[200:].reset_index(drop=True)
    trimmed_endpoint = type("Endpoint", (), {
        "record_index": 390, "Date": long_group.loc[590, "Date"]
    })()
    trimmed_window = build_feature_row(trimmed_endpoint, trimmed, 12)
    check("old_out_of_grid_rows_do_not_change_window",
          same_features(full_window, trimmed_window))

    # 5c. A representação current não contém valores brutos de bins nem resumo.
    current_columns = feature_columns(12, "current", True)
    summary_columns = feature_columns(12, "summary", True)
    check("current_has_no_raw_bins",
          not any(column.startswith("bin") or "__last__" in column
                  for column in current_columns))
    check("current_has_no_summary",
          not any(column.startswith("dmean__") or column.startswith("dstd__")
                  or column.startswith("dslope__")
                  for column in current_columns))
    check("summary_adds_only_summary_stats",
          set(current_columns).issubset(set(summary_columns))
          and len(set(summary_columns) - set(current_columns)) == 24)
    check("feature_revision_is_v2", FEATURE_CODE_VERSION.endswith("v2"))

    # 6. Inclinação por dia real.
    day = pd.Timestamp("2024-01-01")
    _, _, slope = _daily_stats(pd.Series([day, day + pd.Timedelta(days=1)]),
                               np.array([0.0, 2.0]))
    check("slope_per_actual_day", abs(slope - 2.0) < 1e-12)

    # 7. Splits disjuntos e cobertura completa.
    check("split_disjoint", all(
        set(row["test_ids"]).isdisjoint(set(row["outer_train_ids"]))
        and set(row["val_ids"]).isdisjoint(set(row["inner_train_ids"]))
        for row in SPLITS.to_dict(orient="records")
    ))

    # 8. Imputador/scaler ajustados só no treino, incluindo coluna toda NaN.
    x_train = pd.DataFrame({
        "cur__s0": [1.0, 2.0, np.nan],
        "cur__s1": [np.nan, np.nan, np.nan],
    })
    preprocessor = TabularPreprocessor("standard").fit(x_train)
    check("all_nan_fallback_constant",
          "cur__s1" in preprocessor.constant_fallback_columns_
          and preprocessor.fill_values_["cur__s1"] == 0.0)
    none_preprocessor = TabularPreprocessor("none").fit(x_train)
    forced = none_preprocessor.transform(pd.DataFrame({
        "cur__s0": [5.0], "cur__s1": [99.0]}))
    check("all_nan_column_stays_constant_on_new_data",
          forced[0, 1] == 0.0 and forced[0, 3] == 0.0)
    mean_before = preprocessor.scaler_.mean_.copy()
    transformed = preprocessor.transform(x_train)
    check("all_nan_transform_finite", np.isfinite(transformed).all())
    check("fixed_missing_indicators",
          len(preprocessor.indicator_columns_) == x_train.shape[1])
    changed = x_train.copy()
    changed.loc[0, "cur__s0"] = 1e9
    preprocessor.transform(changed)
    check("scaler_not_refit_on_transform",
          np.array_equal(mean_before, preprocessor.scaler_.mean_))
    check("signed_log_keeps_negative",
          signed_log1p(np.array([-3.0, 3.0]))[0] < 0)

    # 9. Pesos: componente e componente-fase, ambos média normalizada.
    ids = np.array([1, 1, 1, 2])
    phases = np.array(["saudavel", "pos_FPT", "pos_FPT", "saudavel"])
    component_weights = compute_weights(ids, phases, "component")
    check("component_weight_mean_one", abs(component_weights.mean() - 1.0) < 1e-12)
    sums = [component_weights[ids == component].sum()
            for component in sorted(set(ids))]
    check("component_weight_equal_components", abs(sums[0] - sums[1]) < 1e-12)
    phase_weights = compute_weights(ids, phases, "component_phase")
    check("component_phase_weight_mean_one", abs(phase_weights.mean() - 1.0) < 1e-12)
    component_sums = [phase_weights[ids == component].sum()
                      for component in sorted(set(ids))]
    check("component_phase_equal_components",
          abs(component_sums[0] - component_sums[1]) < 1e-12)
    for component in sorted(set(ids)):
        cell_sums = [
            phase_weights[(ids == component) & (phases == phase)].sum()
            for phase in sorted(set(phases[ids == component]))
        ]
        check(f"component_phase_cells_equal_{component}",
              max(cell_sums) - min(cell_sums) < 1e-12)

    # 10. Unidades nativas e ausência de clamp.
    day_native, day_metric = scaled_prediction_to_units(np.array([0.5]), "dias-90")
    fraction_native, fraction_metric = scaled_prediction_to_units(
        np.array([0.5]), "porcentagem-data")
    negative_native, _ = scaled_prediction_to_units(np.array([-0.2]), "dias-90")
    check("day_units", abs(day_native[0] - 50.0) < 1e-12
          and abs(day_metric[0] - 50.0) < 1e-12)
    check("fraction_units_pp", abs(fraction_native[0] - 0.5) < 1e-12
          and abs(fraction_metric[0] - 50.0) < 1e-12)
    check("no_clamp_negative", negative_native[0] < 0)
    plot_y, plot_pred, plot_unit = plot_metric_columns("porcentagem-data")
    check("plot_uses_metric_columns",
          plot_y == "y_metric" and plot_pred == "pred_metric"
          and "pontos percentuais" in plot_unit)
    plot_target = pd.DataFrame({
        "endpoint_id": ["F1"], "Falha": [1], "equipment": ["A"],
        "Date": [pd.Timestamp("2024-01-01")], "phase": ["saudavel"],
        "RUL": [0.5], "y_metric": [50.0],
    })
    plot_frame = predictions_frame(
        plot_target, np.array([0.55]), "porcentagem-data",
        {"fold_id": "fold_A", "test_equipment": "A"},
        {"candidate_id": "plot", "family": "mean"},
    )
    check("plot_values_are_pp",
          abs(plot_frame["y_metric"].iloc[0] - 50.0) < 1e-12
          and abs(plot_frame["pred_metric"].iloc[0] - 55.0) < 1e-12)

    # 10b. Macro entre folds não mistura famílias.
    macro_input = pd.DataFrame([
        {"label": "dias-90", "family": "mean", "unit": "dias",
         "fold_id": "fold_A", "test_equipment": "A", "endpoint_id": "m1",
         "Falha": 1, "equipment": "A", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 12.0},
        {"label": "dias-90", "family": "mean", "unit": "dias",
         "fold_id": "fold_B", "test_equipment": "B", "endpoint_id": "m2",
         "Falha": 2, "equipment": "B", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 8.0},
        {"label": "dias-90", "family": "ridge", "unit": "dias",
         "fold_id": "fold_A", "test_equipment": "A", "endpoint_id": "r1",
         "Falha": 1, "equipment": "A", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 14.0},
        {"label": "dias-90", "family": "ridge", "unit": "dias",
         "fold_id": "fold_B", "test_equipment": "B", "endpoint_id": "r2",
         "Falha": 2, "equipment": "B", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 6.0},
    ])
    macro = aggregate_metric_rows(fold_metric_rows(macro_input))
    mean_macro = macro[(macro["family"] == "mean")
                       & (macro["scope"] == "overall")
                       & (macro["name"] == "all")]["mae"].iloc[0]
    ridge_macro = macro[(macro["family"] == "ridge")
                        & (macro["scope"] == "overall")
                        & (macro["name"] == "all")]["mae"].iloc[0]
    check("macro_keeps_families_separate",
          abs(mean_macro - 2.0) < 1e-12 and abs(ridge_macro - 4.0) < 1e-12)

    # 10c. Completude esperada de payloads por família.
    expected_payload_count = len(LABEL_ORDER) * len(SPLITS) * len(MODEL_FAMILIES)
    check("expected_payload_count", expected_run_count() == expected_payload_count)
    payload_paths = {
        run_file_path(label, fold["fold_id"], family)
        for label in LABEL_ORDER
        for fold in SPLITS.to_dict(orient="records")
        for family in MODEL_FAMILIES
    }
    check("payload_paths_include_family", len(payload_paths) == expected_payload_count)
    check("eda_sensor_rows", len(SENSOR_EDA) == len(SENSOR_COLUMNS))
    check("eda_gap_rows", len(TIME_GAP_EDA) == len(FAULT_IDS))

    # 11. Fingerprint e retomada atômica.
    check("fingerprint_order_independent",
          sha256_json({"a": 1, "b": 2}) == sha256_json({"b": 2, "a": 1}))
    check("fingerprint_changes",
          sha256_json({"a": 1}) != sha256_json({"a": 2}))
    with tempfile.TemporaryDirectory(dir=str(CACHE_DIR)) as temporary:
        temporary = Path(temporary)
        path = temporary / "run.json"
        payload = {
            "run_identity": "id-teste",
            "label": "dias-90",
            "family": "mean",
            "fold_id": "fold_A",
            "candidate_mode": "quick",
            "chosen_candidate": {"candidate_id": "abc"},
            "metrics": {"overall": {"mae": 1.0}},
            "counts": {"n_outer_test": 1},
            "predictions": [{"endpoint_id": "F01-r0005"}],
        }
        check("resume_fixture_matches_required_schema",
              set(payload) == set(RUN_REQUIRED_KEYS))
        save_complete_run(path, payload)
        loaded = load_complete_run(path, "id-teste", ["F01-r0005"])
        check("resume_complete_load", loaded is not None
              and len(loaded["predictions"]) == 1)
        broken = dict(payload)
        broken.pop("predictions")
        atomic_write_json(path, seal_payload(broken))
        try:
            load_complete_run(path, "id-teste", ["F01-r0005"])
            raise AssertionError("JSON sem predições deveria ser rejeitado")
        except RunIntegrityError:
            results.append("resume_missing_predictions_rejected")
        save_complete_run(path, payload)
        tampered = read_json(path)
        tampered["metrics"]["overall"]["mae"] = 999.0
        atomic_write_json(path, tampered)
        try:
            load_complete_run(path, "id-teste", ["F01-r0005"])
            raise AssertionError("JSON adulterado deveria falhar na integridade")
        except RunIntegrityError:
            results.append("resume_bad_integrity_rejected")

    # 12. Nenhuma feature com nomes proibidos.
    for width in PROTOCOL["bin_hours_candidates"]:
        for representation in REPRESENTATIONS:
            for column in feature_columns(width, representation, True):
                if any(token in column for token in FORBIDDEN_INFERENCE_TOKENS):
                    raise AssertionError(f"feature proibida: {column}")
    results.append("no_forbidden_feature_names")

    print(f"OK — {len(results)} verificações leves passaram.")
    return results


SELF_TEST_RESULTS = run_self_tests()


## Execução

A célula seguinte é a única que dispara treinos. Ela:

1. cria o diretório único da execução;
2. exporta auditoria, EDA, endpoints e splits;
3. imprime o **preview de ajustes antes de qualquer treino**;
4. em `run_training=False`, encerra sem ajustar modelos;
5. em `run_training=True`, executa **4 famílias × 4 folds × 5 rótulos = 80 payloads**, refaz o teste externo de cada família somente após a respectiva seleção interna e exporta predições, métricas e ZIP.

Para habilitar o treino, altere na célula de configuração:

```python
CONFIG["run_training"] = True
```

Depois **REEXECUTE TODAS AS CÉLULAS** (`Runtime > Restart and run all`). Como `EFFECTIVE_CONFIG` e `RUN_ID` são derivados de `CONFIG`, executar apenas a configuração e a última célula deixa a identidade obsoleta e pode reutilizar/colidir com resultados da execução anterior.

A média (`mean`) é uma família independente e é sempre refeita/testada. Não há escolha de uma família vencedora pelo teste externo.


In [ ]:
# =============================================================================
# SEÇÃO 16 — EXECUÇÃO
# =============================================================================

started_utc = RUNTIME_STARTED_UTC
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
export_data_tables()

preview = estimate_expected_fits()
print_fit_preview(preview)

RUN_PAYLOADS = []
ABLATION_RESULTS = {}
FIT_COUNTER = {}

if not CONFIG["run_training"]:
    finished_utc = datetime.now(timezone.utc).isoformat()
    RESULTS_EXPORT = export_results(
        payloads=[], preview=preview, fit_counter=FIT_COUNTER,
        started_utc=started_utc, finished_utc=finished_utc,
        ablation_results={},
    )
    print("\nSMOKE CONCLUÍDO SEM TREINO.")
    print("Para habilitar os ajustes, mude CONFIG['run_training']=True e")
    print("reexecute TODAS as células (Runtime > Restart and run all).")
    print("Reexecutar apenas a configuração e esta célula deixa RUN_ID obsoleto.")
    print("ZIP:", RESULTS_EXPORT["zip_path"])
else:
    def get_features(width_hours):
        return get_feature_matrix(width_hours)

    for label in LABEL_ORDER:
        for fold in SPLITS.to_dict(orient="records"):
            for family in MODEL_FAMILIES:
                payload, reused = execute_label_fold_family(
                    label, fold, family, get_features, FIT_COUNTER
                )
                RUN_PAYLOADS.append(payload)

    if CONFIG["ablation"]["enabled"]:
        ABLATION_RESULTS = run_ablation(get_features, FIT_COUNTER)

    actual_total = int(FIT_COUNTER.get("model_fits", 0))
    actual_mean_selection = int(FIT_COUNTER.get("selection_baseline_fits", 0))
    actual_mean_final = int(FIT_COUNTER.get("final_baseline_fits", 0))
    if len(RUN_PAYLOADS) != expected_run_count():
        raise RuntimeError(
            f"Saídas incompletas: {len(RUN_PAYLOADS)} payloads, "
            f"esperado {expected_run_count()}."
        )
    if actual_total != int(preview["total_model_fits"]):
        raise RuntimeError(
            f"Contagem de ajustes divergente: preview={preview['total_model_fits']} "
            f"atual={actual_total}."
        )
    if actual_mean_selection != int(preview["mean_selection_fits"]):
        raise RuntimeError(
            "Contagem de ajustes de seleção da média divergente: "
            f"preview={preview['mean_selection_fits']} atual={actual_mean_selection}."
        )
    if actual_mean_final != int(preview["mean_final_fits"]):
        raise RuntimeError(
            "Contagem de refits finais da média divergente: "
            f"preview={preview['mean_final_fits']} atual={actual_mean_final}."
        )

    finished_utc = datetime.now(timezone.utc).isoformat()
    RESULTS_EXPORT = export_results(
        payloads=RUN_PAYLOADS, preview=preview, fit_counter=FIT_COUNTER,
        started_utc=started_utc, finished_utc=finished_utc,
        ablation_results=ABLATION_RESULTS,
    )
    print("\nTreino concluído.")
    print("Payloads por família:", {
        family: sum(payload["family"] == family for payload in RUN_PAYLOADS)
        for family in MODEL_FAMILIES
    })
    print("Ajustes efetivos:", actual_total,
          "| seleção média:", actual_mean_selection,
          "| refit final média:", actual_mean_final)
    print("ZIP:", RESULTS_EXPORT["zip_path"])
    display(RESULTS_EXPORT["metrics_macro"].head(40))


## Leitura responsável e diferenças em relação ao material original

### O que o protocolo antigo fazia

- O notebook de origem (`RUL_CMS_TCN.ipynb`, célula 33) avaliava **sete tarefas de seis IDs de treino e um ID de teste**, uma por Falha. Isso não é o mesmo que generalizar para um equipamento físico inteiro.
- Na preparação de validação (célula 14), o período pós-FPT era **truncado**: mantinha-se o trecho saudável e apenas a primeira metade do período de falha ativa. Assim, o teste não cobria o ciclo de falha completo.
- Havia normalização MinMax **local por série**, ajustada também nos dados de teste/validação (célula 14 e variantes de normalização pareada nas células 15–17). Isso usa a faixa do próprio teste e não é uma escala aprendida apenas no treino.
- Em versões legadas, `evaluate_model` (célula 29) aplicava `torch.clamp(predictions, 0.0, 1.0)` antes de calcular as métricas. Esse clip é adequado a rótulos percentuais (fração 0..1), mas **não** a rótulos de dias, cujo alvo vai até 90/100 dias: na escala de dias, `[0,1]` reduziria qualquer predição acima de 1 dia. Neste notebook **não há clip em nenhum rótulo**: predições de dias e de percentual podem sair do intervalo observado.
- O material original também calculava MAPE e usava outras escolhas de métrica. Aqui MAPE não é calculado, não há clamp e não há comparação de score entre rótulos.

### O protocolo novo

- Holdout de **equipamento físico**: `A=[1,2]`, `B=[3]`, `C=[4,5]`, `D=[6,7]`; quatro folds leave-one-equipment-out.
- Um equipamento interno de validação, dois de treino interno e três de treino externo. Existe **apenas um** equipamento de validação interna por fold — uma limitação real da estimativa de seleção.
- Endpoints somente após 30 registros de aquecimento, passo 3 e endpoint final obrigatório. **Não** se afirma cobertura de todas as linhas originais; a cobertura é o ciclo pós-aquecimento.
- Features causais por endpoint, com 32 bins regulares `(left, right]`: o valor corrente e o resumo por dia real vêm da mesma janela. Valores brutos dos bins não entram na matriz tabular; nenhum dado futuro, nenhum `ffill`, nenhuma interpolação e nenhuma mistura entre Falhas.
- Imputação, escala e seleção de configuração são aprendidas por família, apenas no treino interno; o refit final de cada família usa o treino externo e o teste externo é avaliado uma vez por família. Nenhuma família é escolhida como vencedora pelo teste externo.
- A referência de média é uma família independente, sempre sem pesos e sempre testada. Pesos opcionais, quando habilitados, existem apenas no treino.
- As métricas “overall” por fold agregam componentes dentro do equipamento de teste. A agregação entre os quatro folds é uma média simples por família, igual entre equipamentos, não uma recombinação pooled das amostras dos folds.
- Datas FPT são **anotações herdadas** e não foram verificadas fisicamente de forma independente. Elas aparecem somente em relatório de fase e pesos opcionais.

### Limitações que devem acompanhar qualquer conclusão

- Quatro máquinas/equipamentos é uma amostra pequena; a média macro entre folds tem apenas quatro unidades e não deve ser lida como estimativa precisa.
- O cenário é generalização para um equipamento não visto, não previsão do futuro do mesmo equipamento.
- O dataset contém Falhas com históricos e regimes distintos; agrupar por equipamento reduz, mas não elimina, heterogeneidade.
- SQLite em Drive tem risco de corrupção com escritores concorrentes. Mantenha um único runtime escrevendo no diretório.
- A execução científica no Colab não foi realizada na criação deste notebook. A verificação local cobriu somente estrutura, sintaxe, esquema/consolidação e inspeção semântica; portanto, métricas e tempos de execução permanecem **UNVERIFIED** até uma execução real.


In [ ]:
# =============================================================================
# INVOCAÇÃO EXPLÍCITA DOS TESTES LEVES NO COLAB
# Pode ser reexecutada depois de qualquer alteração de helper.
# =============================================================================

SELF_TEST_RESULTS_AGAIN = run_self_tests()
print("Testes leves reexecutados:", len(SELF_TEST_RESULTS_AGAIN))
